
# Accenture Project Overview

## Challenge Objective

Analyse pollutant levels in European cities using 10 years of annual air quality statistics (2015-2024) published by the European Environment Agency (EEA).
The ultimate goal of the full project is to predict future Air Quality Index (AQI) values or flag regulatory limit exceedances using machine learning.


---
# Environment and Data Loading


10 separate CSV files, one per year between 2015-2024, loaded into a single DataFrame. After loading, we make the first inspection to understand the structure of data.

## Importing Libraries

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.io as pio

# Force white background and light gray grid globally for PyCharm
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'grid.color': '#CCCCCC',
    'grid.alpha': 0.5
})
sns.set_style('whitegrid', rc={'figure.facecolor': 'white', 'axes.facecolor': 'white', 'grid.color': '#CCCCCC'})

# Show all Plotly charts inline in the notebook output
pio.renderers.default = "notebook"


# Global variables
focus_pollutants = ["NO2", "PM10", "PM2.5", "O3"]
secondary_pollutants = ["SO2", "CO", "C6H6"]
all_pollutants = focus_pollutants + secondary_pollutants

aggregation_id = "P1Y-day-per50"

print("Libraries loaded")

## Combining CSV

In [ ]:
DATA = "2015-2024"
years = range(2015, 2025)

frames = []
for year in years:
    file_path = os.path.join(DATA, f"DataExtract{year}.csv")
    df_year = pd.read_csv(file_path, low_memory=False)
    frames.append(df_year)
    print(f"{year}: {df_year.shape[0]:,} rows")

df = pd.concat(frames, ignore_index=True)
print(f"\nCombined DataFrame: {df.shape[0]:,} rows, {df.shape[1]} columns")

DATA_DIR = '2015-2024'
years = list(range(2015, 2025))

## Country comparison dataset

In [ ]:
comparison = "country_comparison"
comparison_folders = ["FR", "CH", "AT"]

df_compare_list = []

for code in comparison_folders:
    folder_path = os.path.join(comparison, code)

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        df_temp = pd.read_csv(file_path, low_memory=False)
        df_compare_list.append(df_temp)
    print(f"{code} loaded.")

# Build focused dataframe directly — no need to store the raw version
df_compare_focus = pd.concat(df_compare_list, ignore_index=True)
df_compare_focus = df_compare_focus[
    (df_compare_focus["Data Aggregation Process Id"] == aggregation_id) &
    (df_compare_focus["Air Pollutant"].isin(focus_pollutants))].copy()

print("df_compare_focus rows:", f"{len(df_compare_focus):,}")
print(df_compare_focus["Country"].value_counts())


## Look at the Raw Data

In [ ]:
df.head(3)

In [ ]:
# Column names and data types
print(df.dtypes)

In [ ]:
# Basic descriptive statistics
df.describe(include='all').T

---
# Data Quality Assessment

*Why this step?*

Before drawing any conclusions, we need to understand how complete and trustworthy the data is.
Data quality problems — missing values, incomplete year coverage, unvalidated entries — can produce misleading charts and wrong conclusions if ignored.
This section answers: can we trust this data, and where are the gaps?


## Missing Values

In [ ]:
# Count missing values per column
missing_count = df.isnull().sum()
miss_percentage = (missing_count / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing_count, 'Missing %': miss_percentage})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('Columns with missing values:')
print(missing_df)
missing_df.to_csv("images/missing_df.csv")

Interpretation

High missingness (~62%) in **City, City Population, and City Code** indicates the need for imputation (e.g., spatial joins) or exclusion. Other variables show minimal missingness (<1%), with **Altitude** nearly complete. Overall, the dataset is largely clean, with only a few key fields requiring targeted preprocessing.  We will address the mising city values by including ISTAT dataset (data boundaries), merging based on coordinates of the cities (long, lat). Regarding the city population we joined two datasets from UN and ISTAT.

In [ ]:
# Bar chart of missing values — Plotly
import plotly.graph_objects as go

fig = go.Figure(go.Bar(
    x=missing_df['Missing %'].values,
    y=missing_df.index.tolist(),
    orientation='h',
    marker_color='steelblue'
))
fig.add_vline(x=5, line_dash='dash', line_color='red',
              annotation_text='5% threshold', annotation_position='top right')
fig.update_layout(
    title='Missing Values by Column',
    xaxis_title='Missing (%)',
    yaxis_title='',
    height=400,
    width=900,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(gridcolor='#CCCCCC'),
    yaxis=dict(gridcolor='#CCCCCC')
)
fig.write_image('images/missing_value_barchart.png', scale=2)
fig.show()

City is missing for ~62% of records. For Italy, we enrich station metadata by spatially joining station coordinates (lat/lon) with ISTAT ‘comuni’ boundaries using GeoPandas. This assigns each station to a municipality (Comune), which we use as City.

## Handling Missing Cities

In [ ]:
# Handling Missing cities
import geopandas as gpd

comuni = gpd.read_file("data_boundaries/comuni_2024/comuni_2024.shp")
print(comuni.crs)
print(comuni.columns.tolist())

# Convert comuni to WGS84 lat/lon to correctly match station coordinates
comuni = comuni.to_crs("EPSG:4326")
print("New CRS:", comuni.crs)

In [ ]:
# Keep Italy rows with valid coordinates
df_it = df[df["Country"] == "Italy"].copy()
df_it = df_it.dropna(subset=["Longitude", "Latitude"])

# Create geometry column from lon/lat
gdf_stations = gpd.GeoDataFrame(
    df_it,
    geometry=gpd.points_from_xy(df_it["Longitude"], df_it["Latitude"]),
    crs="EPSG:4326"
)

print("Station CRS:", gdf_stations.crs)
print("Number of Italy station rows:", len(gdf_stations))

In [ ]:
joined = gpd.sjoin(
    gdf_stations[["Air Quality Station EoI Code", "geometry"]],
    comuni[["COMUNE", "PRO_COM", "geometry"]],
    how="left",
    predicate="within"
)

print("Joined rows:", len(joined))
print("Share of stations matched to a comune:",
      round(joined["COMUNE"].notna().mean() * 100, 2), "%")

joined.head(5).to_csv("images/joined_comune.csv")
joined.head(5)

In [ ]:
# Build mapping from station code to comune name
station_to_city = (
    joined.dropna(subset=["COMUNE"])
    .groupby("Air Quality Station EoI Code")["COMUNE"]
    .agg(lambda s: s.mode().iloc[0])
)

print("Stations with assigned comune:", len(station_to_city))

In [ ]:
# Clean City column first
df["City"] = df["City"].astype("string").str.strip()
df.loc[df["City"] == "", "City"] = pd.NA

it_mask = df["Country"] == "Italy"

before = df.loc[it_mask, "City"].isna().mean() * 100

df.loc[it_mask, "City"] = df.loc[it_mask, "City"].fillna(
    df.loc[it_mask, "Air Quality Station EoI Code"].map(station_to_city)
)

after = df.loc[it_mask, "City"].isna().mean() * 100

print(f"Italy missing City — before: {before:.2f}% | after: {after:.2f}%")
print("Unique cities in full df:", df["City"].nunique())

Interpretation

To address the high missingness in the **City** column, we leveraged geospatial data from Italian *comuni* boundaries. After aligning coordinate systems, air quality monitoring stations were converted into a GeoDataFrame and spatially joined with the comuni polygons.

The join achieved a very high match rate (99.42%), indicating strong geographic alignment between station coordinates and administrative boundaries. Using this mapping, we created a station-to-city reference and used it to impute missing city values.

As a result, missingness in the **City** column was reduced from **62.05% to 0.37%**, significantly improving data completeness. This enables more reliable spatial analysis and allows the inclusion of location-based features in downstream modeling.

## Population Check

Before analysing data coverage, we inspect the original EEA `City Population` field.

For the Italian subset, this variable is often missing and, when available, usually remains unchanged across years for the same city. This makes it unsuitable for studying how exposure evolves over time, which is why we later merge an external yearly population dataset.

In [ ]:
# Check whether the original EEA City Population changes over time
# Focus on Italy because the later population merge is done for Italy

pop_check = df[
    (df["Country"] == "Italy") &
    df["City"].notna() &
    df["Year"].notna()
].copy()

pop_check["City"] = pop_check["City"].astype(str).str.strip()

# One representative population value per City-Year
city_year_pop = (
    pop_check.dropna(subset=["City Population"])
    .groupby(["City", "Year"])["City Population"]
    .median()
    .reset_index()
)

# For each city, count how many distinct population values appear across years
city_pop_variation = (
    city_year_pop.groupby("City")["City Population"]
    .nunique()
    .reset_index(name="n_unique_population_values")
    .sort_values(["n_unique_population_values", "City"])
)

n_cities = len(city_pop_variation)
n_constant = (city_pop_variation["n_unique_population_values"] == 1).sum()
share_constant = n_constant / n_cities if n_cities > 0 else np.nan

print(f"Italian cities with non-missing EEA population info: {n_cities}")
print(f"Cities where EEA population stays exactly the same across all available years: {n_constant}")
print(f"Share of constant-population cities: {share_constant:.1%}")

display(city_pop_variation.head(15))
display(city_pop_variation["n_unique_population_values"].value_counts().sort_index())

Interpretation

The original EEA **City Population** field shows no variation over time: all 81 Italian cities have constant values across years. This indicates the variable is static and unsuitable for temporal analysis, motivating the integration of an external dataset with yearly population dynamics.

In [ ]:
# Show a few example cities
example_cities = ["Roma", "Milano", "Napoli", "Torino", "Bologna"]

example_df = city_year_pop[city_year_pop["City"].isin(example_cities)].copy()

plt.figure(figsize=(8, 4.5))
for city in sorted(example_df["City"].unique()):
    sub = example_df[example_df["City"] == city].sort_values("Year")
    plt.plot(sub["Year"], sub["City Population"], marker="o", label=city)

plt.title("EEA City Population by year: example Italian cities")
plt.xlabel("Year")
plt.ylabel("City Population")
plt.legend()
plt.tight_layout()
plt.savefig("images/eea_population_examples.png", dpi=300)
plt.show()

Because the EEA city population field is mostly static across years, we next enrich the dataset with an external yearly population source to analyse demographic exposure more reliably.

Interpretation

The visualization confirms that the EEA **City Population** field remains constant over time for all example cities, showing no temporal variation. This reinforces that the variable is static and unsuitable for longitudinal analysis, motivating the integration of an external dataset with yearly population dynamics to better capture demographic changes.

## Population dataset joining

#### UN Population Dataset (2015–2024)

The EEA "City Population" field is often missing or constant over time, so we cannot study how exposure varies with population or compute per‑capita metrics reliably.

We load city-level population data from UN statistics (`UNdata_Export_20260312_103717571.csv`) for Italian cities from the United Nations' Statistical Yearbooks (https://data.un.org/Data.aspx?q=city+population&d=POP&f=tableCode%3a240#POP).

We use the UN World Urbanization Prospects city population dataset (Italy, by city and year): load it, normalize city names for matching (case‑insensitive), and fill missing years with the smaller of previous/next year (conservative for concentration per person). Then we merge city–year data and format city names for display (each city starting with a capital letter).

As a fallback, we're using ISTAT population data for 2024 to fill in missing values that were present in EEA dataset and couldn't be handled by UN data.

In [ ]:
# UN Population (primary source)
UN_POP_PATH = "UNdata_Export_20260312_103717571.csv"

un_raw = pd.read_csv(UN_POP_PATH)
un_pop = (
    un_raw.loc[
        (un_raw["Country or Area"] == "Italy") & (un_raw["Sex"] == "Both Sexes"),
        ["Year", "City", "Value"],
    ]
    .rename(columns={"Value": "Population_UN"})
    .copy()
)
un_pop["Year"] = un_pop["Year"].astype(int)

def normalize_city_for_merge(s):
    if pd.isna(s):
        return s
    return str(s).strip().lower()

un_pop["City_norm"] = un_pop["City"].apply(normalize_city_for_merge)

years_full = list(range(2015, 2025))
filled = []
for (city_norm, city_orig), grp in un_pop.groupby(["City_norm", "City"]):
    by_year = grp.set_index("Year")["Population_UN"]
    years_in_grp = sorted(by_year.index)
    for y in years_full:
        if y in by_year.index:
            filled.append({"City_norm": city_norm, "City": city_orig, "Year": y, "Population_UN": by_year.loc[y]})
        else:
            prev_y = [vy for vy in years_in_grp if vy < y]
            next_y = [vy for vy in years_in_grp if vy > y]
            prev_val = by_year.loc[prev_y[-1]] if prev_y else np.nan
            next_val = by_year.loc[next_y[0]] if next_y else np.nan
            if pd.notna(prev_val) and pd.notna(next_val):
                pop_fill = min(prev_val, next_val)
            elif pd.notna(prev_val):
                pop_fill = prev_val
            elif pd.notna(next_val):
                pop_fill = next_val
            else:
                continue
            filled.append({"City_norm": city_norm, "City": city_orig, "Year": y, "Population_UN": pop_fill})

un_pop_filled = pd.DataFrame(filled)
print(f"UN population rows (filled): {len(un_pop_filled)}, years: {sorted(un_pop_filled['Year'].unique())}")

# ISTAT Population (fallback source)
ISTAT_POP_PATH = "ISTAT_pop2024.csv"

istat_raw = pd.read_csv(ISTAT_POP_PATH, low_memory=False)

istat_pop = (
    istat_raw[
        (istat_raw["Territory"] != "Italy") &
        (istat_raw["TIME_PERIOD"] == "2024-12")]
    [["Territory", "Observation"]]
    .rename(columns={"Territory": "City_ISTAT", "Observation": "Population_ISTAT"})
    .dropna(subset=["Population_ISTAT"])
    .copy()
)

istat_pop["City_norm"] = istat_pop["City_ISTAT"].apply(normalize_city_for_merge)

print(f"ISTAT comuni loaded: {len(istat_pop)}")

In [ ]:
print("UN population (filled): years", sorted(un_pop_filled["Year"].unique()))
firenze_pop = (
    un_pop_filled[un_pop_filled["City_norm"] == "milano"]
    [["Year", "City", "Population_UN"]]
    .sort_values("Year")
)
print('Example: population in Milano over 2015-2024')
display(firenze_pop)

### Merge with EEA city–year and city display

We build one row per (City, Year) from EEA (median pollutant, one Lat/Lon per city), merge on normalized city name and year with UN population, then apply **title case** to city names so every city starts with a capital letter in maps and tables.

In [ ]:
aggregation_id = "P1Y-day-per50"
focus_pollutants = ["NO2", "PM10", "PM2.5", "O3"]
secondary_pollutants = ["SO2", "CO", "C6H6"]
all_pollutants = focus_pollutants + secondary_pollutants

df_focus = df[
    (df["Data Aggregation Process Id"] == aggregation_id) &
    (df["Air Pollutant"].isin(focus_pollutants))
].copy()

df_copy = df.copy()


In [ ]:
coords = (
    df_focus.dropna(subset=["City", "Latitude", "Longitude"])
    .groupby("City")[["Latitude", "Longitude"]]
    .mean()
    .reset_index()
)

city_year_pol = (
    df_focus.dropna(subset=["City"])
    .groupby(["City", "Year", "Air Pollutant"])["Air Pollution Level"]
    .median()
    .reset_index()
)

city_pivot = city_year_pol.pivot_table(
    index=["City", "Year"],
    columns="Air Pollutant",
    values="Air Pollution Level",
    aggfunc="median",
).reset_index()

pol_cols = [p for p in focus_pollutants if p in city_pivot.columns]

by_city_year = city_pivot.merge(coords, on="City", how="left")
by_city_year["City_norm"] = by_city_year["City"].apply(normalize_city_for_merge)
by_city_year["Year"] = by_city_year["Year"].astype(int)

# Baseline: raw EEA missing
_raw_missing = df["City Population"].isna().sum()
_raw_total = len(df)
print(f"Raw EEA — City Population missing: {_raw_missing} / {_raw_total} ({_raw_missing/_raw_total:.1%})")

# UN primary
merged = by_city_year.merge(
    un_pop_filled[["City_norm", "Year", "Population_UN"]],
    on=["City_norm", "Year"],
    how="left",
)

# ISTAT fallback
merged = merged.merge(
    istat_pop[["City_norm", "Population_ISTAT"]],
    on="City_norm",
    how="left",
)

merged["City_Population"] = merged["Population_UN"].fillna(merged["Population_ISTAT"])
merged.drop(columns=["Population_ISTAT"], inplace=True)

def city_display(s):
    if pd.isna(s):
        return s
    return str(s).strip().lower().title()

merged["City"] = merged["City"].apply(city_display)

print("Merged city-year shape:", merged.shape)
display(merged[merged["City"] == "Roma"].sort_values("Year"))

In [ ]:
total = len(merged)
un_filled = merged["Population_UN"].notna().sum()
istat_added = (merged["Population_UN"].isna() & merged["City_Population"].notna()).sum()
total_filled = merged["City_Population"].notna().sum()
still_missing = merged["City_Population"].isna().sum()

print(f"Raw EEA — City Population missing:    {_raw_missing} / {_raw_total} ({_raw_missing/_raw_total:.1%})")
print(f"Filled by UN:                         {un_filled:<6}  ({un_filled/total:.1%})")
print(f"Additionally filled by ISTAT:         {istat_added:<6}  ({istat_added/total:.1%})")
print(f"Total filled (UN + ISTAT):            {total_filled:<6}  ({total_filled/total:.1%})")
print(f"Still missing after both sources:     {still_missing:<6}  ({still_missing/total:.1%})")
print()

if still_missing > 0:
    missing_cities = (
        merged[merged["City_Population"].isna()]["City"]
        .value_counts()
        .reset_index()
    )
    missing_cities.columns = ["City", "Rows_missing"]
    print(f"Cities still missing ({len(missing_cities)} cities):")
    display(missing_cities)

Interpretation

To address missing and static population data, we integrate UN city-level population (2015–2024) as the primary source, filling gaps conservatively using neighboring years. ISTAT 2024 data is used as a fallback for unmatched cities. After merging on normalized city names and year, population coverage improves substantially: 96.5% of missing values are filled, leaving only 3.5% unresolved. This enriched dataset enables reliable per-capita and temporal analysis of air pollution exposure.

## Data Coverage

The Data Coverage column tells us what fraction of the year a station was actually measuring.
A station with 40% coverage was only active for about 5 months — its annual summary is less reliable.
The EEA recommends at least 75% coverage for a statistic to be considered valid.

In [ ]:
# Distribution of Data Coverage
df['Data Coverage'].dropna().plot(kind='hist', bins=40, figsize=(9, 4), color='steelblue', edgecolor='white')
plt.axvline(75, color='red', linestyle='--', label='75% threshold')
plt.xlabel('Data Coverage (%)')
plt.title('Distribution of Data Coverage')
plt.legend()
plt.tight_layout()
plt.savefig("images/distribution_of_data_coverage.png", dpi=300)
plt.show()

above_75 = (df['Data Coverage'] >= 75).sum()
below_75 = (df['Data Coverage'] < 75).sum()
print(f'Coverage >= 75%: {above_75:} rows ({above_75 / len(df) * 100:.1f}%)')
print(f'Coverage <  75%: {below_75:} rows ({below_75 / len(df) * 100:.1f}%)')

Interpretation

Most observations meet the EEA reliability threshold, with 90.4% of rows having ≥75% data coverage. Only 9.6% fall below this level, indicating generally high data quality, though low-coverage observations may introduce noise and should be treated cautiously.

## Verification Flags

The Verification column records whether data was officially validated by the reporting authority.
Value 1 means validated. Anything else should be treated with caution.

In [ ]:
verif_counts = df['Verification'].value_counts(dropna=False)
print(verif_counts)

verif_counts.plot(kind='bar', figsize=(6, 4), color='steelblue', edgecolor='white')
plt.title('Verification Status of Records')
plt.xlabel('Verification Code')
plt.ylabel('Number of Records')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("images/verification_status.png", dpi=300)
plt.show()

## Aggregation Process Breakdown

Each row does not represent a direct sensor reading. It represents a statistical summary of an entire year.
Different rows for the same station and year represent different statistics:
annual median, annual maximum, annual 99th percentile, and so on.

This matters a lot: mixing different aggregation types in one analysis would be like comparing apples and oranges.
For the EDA sections that follow, we will consistently use the annual median (P1Y-day-per50),
which is the 50th percentile of daily values over a full year.

In [ ]:
agg_counts = df['Data Aggregation Process'].value_counts().head(15)
agg_counts.to_csv("images/top_15_aggregation_processes.csv")
print(agg_counts.to_string())

agg_counts.plot(kind='barh', figsize=(11, 5), color='steelblue')
plt.xlabel('Number of Records')
plt.title('Top 15 Aggregation Process Types in the Dataset')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("images/top_15_aggregation_processes.png", dpi=300)
plt.show()

## Duplicate Check

In [ ]:
# Exact duplicate rows
n_dup = df.duplicated().sum()
print(f'Exact duplicate rows: {n_dup:,} ({n_dup / len(df) * 100:.2f}%)')

# Logical duplicates: same station + pollutant + aggregation type + year
key_cols = ['Air Quality Station EoI Code', 'Air Pollutant', 'Data Aggregation Process', 'Year']
n_logical = df.duplicated(subset=key_cols).sum()
print(f'Logical duplicate rows (same station, pollutant, aggregation, year): {n_logical:,}')

## Conclusion

Key data quality observations:

- Which columns have missing values and at what proportion
- What fraction of rows have low data coverage (below 75%) — these produce less reliable annual statistics
- How many records are unverified — these may need to be excluded from modeling
- The dataset contains multiple aggregation types per station — analysis must group by type or fix one type


---
# EDA: Pollutant Distributions

*Why this step?*

Before comparing across time or geography, we need to understand what each pollutant looks like on its own.
Analysis answers:

- Which pollutants are most commonly measured in the dataset?
- What is the typical range of pollution levels across Italian stations?
- Are there extreme outliers? Is the distribution skewed?

From here onward we work with annual median values only (aggregation code P1Y-day-per50).
This gives us one representative number per station per year, making comparisons fair and consistent.


## Which pollutants are present?

In [ ]:
pollutant_counts = df['Air Pollutant'].value_counts()
pollutant_counts.to_csv('images/pollutant_counts.csv')
print(pollutant_counts.to_string())

# Plotly bar chart
fig = go.Figure(go.Bar(
    x=pollutant_counts.index.tolist(),
    y=pollutant_counts.values,
    marker_color='steelblue',
    marker_line_color='white',
    marker_line_width=0.5
))
fig.update_layout(
    title='Number of Records per Pollutant (all years, all aggregation types)',
    xaxis_title='Pollutant',
    yaxis_title='Records',
    xaxis_tickangle=-30,
    height=400,
    width=900,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(gridcolor='#CCCCCC'),
    yaxis=dict(gridcolor='#CCCCCC')
)
fig.write_image('images/pollutant_counts.png', scale=2)
fig.show()

The dataset contains a large number of pollutants, but their observation frequency varies widely.

For the main analysis we focus on the four most consistently monitored pollutants (NO2, PM10, PM2.5, O3).

However, three additional pollutants (SO2, CO, C6H6) are included to enrich the interpretation of emission sources.

## Focused DataFrame

In [ ]:
# Create df_focus: EDA subset (annual median-of-daily values + main pollutants)
df_focus = df[
    (df["Data Aggregation Process Id"] == aggregation_id) &
    (df["Air Pollutant"].isin(focus_pollutants))].copy()

# Create df_extended: includes both main and secondary pollutants
df_extended = df[
    (df["Data Aggregation Process Id"] == aggregation_id) &
    (df["Air Pollutant"].isin(all_pollutants))
]

print("df_focus rows:", f"{len(df_focus):,}")
print("Countries in focus:", df_focus["Country"].nunique())
print(df_focus["Country"].value_counts())

print("\nPollutant summary (Air Pollution Level):")
print(df_focus.groupby("Air Pollutant")["Air Pollution Level"].describe().round(2))

Interpretation

The analysis focuses exclusively on Italy, with key pollutants showing distinct distributions. O₃ has the highest average levels, while PM2.5 is lowest. Variability is greatest for NO₂ and O₃, indicating differing emission dynamics.

In [ ]:
# Checking unit of air pollution for chosen pollutants
pollutant_units = (df_extended[['Air Pollutant', 'Unit Of Air Pollution Level']]
                   .drop_duplicates())            # remove duplicates)

display(pollutant_units)
pollutant_units.to_csv("images/pollutant_units.csv")

Interpretation

Most pollutants are measured in μg/m³, ensuring consistency across variables. CO is reported in mg/m³, requiring unit conversion if included in joint analysis to maintain comparability.

## Histograms per pollutant

In [ ]:
# Histograms — one per pollutant — Plotly
from plotly.subplots import make_subplots
import plotly.graph_objects as go

colors_plotly = ['steelblue', 'tomato', 'goldenrod', 'mediumseagreen']
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'{p} — Distribution of Annual Median' for p in focus_pollutants])

for idx, (pollutant, color) in enumerate(zip(focus_pollutants, colors_plotly)):
    row, col = divmod(idx, 2)
    data = df_focus[df_focus['Air Pollutant'] == pollutant]['Air Pollution Level'].dropna()
    cap = data.quantile(0.99)
    data_capped = data[data <= cap]
    median_val = data.median()
    mean_val = data.mean()

    fig.add_trace(
        go.Histogram(x=data_capped, nbinsx=50, marker_color=color,
                     marker_line_color='white', marker_line_width=0.5,
                     name=pollutant, showlegend=False),
        row=row+1, col=col+1
    )
    fig.add_vline(x=median_val, line_dash='dash', line_color='black',
                  annotation_text=f'<b>Median</b>: {median_val:.1f}',
                  annotation_position='top left',
                  annotation_font=dict(size=11, color='black', family='Arial Black'),
                  annotation_yshift=25,
                  row=row+1, col=col+1)
    fig.add_vline(x=mean_val, line_dash='dot', line_color='red',
                  annotation_text=f'<b>Mean</b>: {mean_val:.1f}',
                  annotation_position='top right',
                  annotation_font=dict(size=11, color='red', family='Arial Black'),
                  annotation_yshift=25,
                  row=row+1, col=col+1)

fig.update_layout(
    title_text='Pollution Level Distributions (Annual Median, raw data)',
    height=700, width=1100,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(t=140),
)
# Place each subplot title on top of its plot (above the plot area)
titles = [f'{p} — Distribution of Annual Median' for p in focus_pollutants]
for i, (px, py) in enumerate([(0.25, 1.08), (0.75, 1.08), (0.25, 0.48), (0.75, 0.48)]):
    fig.layout.annotations[i].update(x=px, y=py, xref='paper', yref='paper', text=titles[i])

fig.update_xaxes(gridcolor='#CCCCCC')
fig.update_yaxes(gridcolor='#CCCCCC')
fig.write_image('images/pollution_level_distributions.png', scale=2)
fig.show()

Interpretation

Pollution levels show right-skewed distributions across all pollutants, with means slightly higher than medians, indicating the presence of high-value outliers. O₃ exhibits the highest concentration levels, while PM2.5 remains lowest. NO₂ and PM10 display moderate variability, reflecting diverse emission sources and urban conditions.

### Extended Pollutant Distributions (Secondary Pollutants)

In [ ]:
extra_colors = ['slateblue', 'darkorange', 'teal']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, pollutant, color in zip(axes, secondary_pollutants, extra_colors):
    data = df_extended[df_extended["Air Pollutant"] == pollutant]["Air Pollution Level"].dropna()

    cap = data.quantile(0.99)
    data_capped = data[data <= cap]

    # compute statistics
    mean_val = data_capped.mean()
    median_val = data_capped.median()

    ax.hist(data_capped, bins=30, color=color, edgecolor='white')

    # mean and median lines
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f"Mean: {mean_val:.2f}")
    ax.axvline(median_val, color='black', linestyle='-', linewidth=2, label=f"Median: {median_val:.2f}")

    unit = df_extended[df_extended["Air Pollutant"] == pollutant]["Unit Of Air Pollution Level"].dropna().mode()
    unit_text = unit.iloc[0] if not unit.empty else ""
    ax.set_title(f"{pollutant} ({unit_text})")
    ax.set_xlabel("Air Pollution Level")
    ax.set_ylabel("Frequency")
    ax.legend()

plt.tight_layout()
plt.savefig("images/secondary_pollution_level_distributions.png", dpi=300)
plt.show()

Interpretation

SO₂, CO, and C6H6 distributions are right-skewed, with means above medians, indicating occasional high values. CO shows tighter spread, while SO₂ and C6H6 exhibit greater variability across observations.

## Boxplots

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(data=df_focus, x='Air Pollutant', y='Air Pollution Level',
            hue='Air Pollutant', legend=False,
            order=focus_pollutants, showfliers=True, flierprops={'marker':'.'},
            palette=['#4c72b0', '#55a868', '#c44e52', '#8172b2'])

plt.title('Overall Distribution of Pollutant Levels (2015-2024)')
plt.ylabel('ug/m3')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig("images/overall_pollution_level_distributions.png", dpi=300)
plt.show()

Interpretation

O₃ shows the highest concentration levels and widest variability, followed by PM10 and NO₂. PM2.5 remains lowest and more stable. All pollutants exhibit outliers, indicating occasional extreme values, particularly for O₃ and NO₂.

## Conclusion

Key takeaways from the pollutant distributions:

- All distributions are right-skewed. Most stations record moderate levels, but a small number of stations are severe hotspots far above the average.
- The gap between median and mean reveals how much those extreme stations pull the average upward.
- Outliers typically represent high-traffic urban cores or industrial areas — the stations regulators care most about.
- This right skew confirms that in the cleaning step, we will need a strategy for handling extreme values before modeling.


---
# EDA: Trends Over 2015-2024

*Why this step?*

Temporal analysis answers the most fundamental policy question: is air quality improving over time?
Three events make this decade interesting:

- COVID-19 lockdowns (2020): traffic and industrial activity nearly stopped — did pollution drop?
- EU Clean Air directives: ongoing regulatory pressure should appear as a gradual downward trend
- Post-COVID rebound (2021-2022): did pollutant levels return once restrictions lifted?

We compute the median pollution level across all stations for each year, which gives a Europe-wide signal.

## Median Polution Level

In [ ]:
# Median pollution level by year for each pollutant
trend = (
    df_focus
    .groupby(['Year', 'Air Pollutant'])['Air Pollution Level']
    .median()
    .reset_index()
)
trend.columns = ['Year', 'Pollutant', 'Median Level']

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.flatten()
colors = ['steelblue', 'tomato', 'goldenrod', 'mediumseagreen']

for i, (pollutant, color) in enumerate(zip(focus_pollutants, colors)):
    sub = trend[trend['Pollutant'] == pollutant].sort_values('Year')
    axes[i].plot(sub['Year'], sub['Median Level'], marker='o', color=color, linewidth=2)
    axes[i].axvspan(2019.5, 2020.5, color='lightcoral', alpha=0.3, label='COVID 2020')
    axes[i].set_title(f'{pollutant} — Annual Median across All Stations')
    axes[i].set_ylabel('ug/m3')
    axes[i].set_xticks(sub['Year'].unique())
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(fontsize=9)

plt.suptitle('Temporal Trends in Air Pollution — Italy 2015-2024', fontsize=13)
plt.tight_layout()
plt.savefig("images/temporal_trend_distributions.png", dpi=300)
plt.show()

Interpretation

Air pollution trends show a clear decline for NO₂, PM10, and PM2.5 from 2015 to 2024, with a sharp drop in 2020 during COVID-19 lockdowns. Levels partially rebound afterward but continue a gradual downward trend. In contrast, O₃ follows a different pattern, increasing post-2020 before slightly decreasing, reflecting distinct atmospheric dynamics.

## Yearly Percentage Change

In [ ]:
YoY = trend.copy()
YoY['YoY Change %'] = YoY.groupby('Pollutant')['Median Level'].pct_change() * 100

# Plotly line chart
colors_plotly = ["steelblue", "tomato", "goldenrod", "mediumseagreen"]

fig = go.Figure()

# COVID shaded band
fig.add_vrect(
    x0=2019.5, x1=2020.5,
    fillcolor="lightcoral", opacity=0.25,
    layer="below", line_width=0,
    annotation_text="COVID 2020",
    annotation_position="top left"
)

# Zero baseline
fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)

for pollutant, color in zip(focus_pollutants, colors_plotly):
    sub = YoY[(YoY['Pollutant'] == pollutant) & YoY['YoY Change %'].notna()]
    fig.add_trace(go.Scatter(
        x=sub['Year'], y=sub['YoY Change %'],
        mode="lines+markers",
        marker=dict(size=7),
        line=dict(color=color, width=2),
        name=pollutant
    ))

fig.update_layout(
    title="Year-over-Year Change in Median Pollution Level (%)",
    xaxis_title="Year",
    yaxis_title="Change (%)",
    height=500, width=1100,
    plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
    xaxis=dict(gridcolor="#CCCCCC", dtick=1),
    yaxis=dict(gridcolor="#CCCCCC"),
    legend=dict(title="Pollutant")
)
fig.write_image("images/YoY_change_in_median.png", scale=2)
fig.show()

# Print the table
(YoY.pivot(index='Year', columns='Pollutant', values='YoY Change %').round(2)
            .to_csv('images/YoY_change_in_median.csv'))
print(YoY.pivot(index='Year', columns='Pollutant', values='YoY Change %').round(2))

Since there is no 2014 data, it's impossible to calculate 2015 change

Interpretation

Year-over-year changes confirm strong volatility around COVID-19. In 2020, NO₂ shows the sharpest drop (−17%), with consistent declines across PM10 and PM2.5, reflecting reduced mobility and activity. A rebound follows in 2021–2022, especially for PM10 and PM2.5. However, 2023–2024 return to negative growth, reinforcing a longer-term downward trend. O₃ remains less stable, with mixed increases and decreases due to different atmospheric behavior.

## Secondary Pollutant Trends

In [ ]:
colors_secondary = {
    "SO2": "slateblue",
    "CO": "darkorange",
    "C6H6": "teal"
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

for ax, pollutant in zip(axes, secondary_pollutants):

    subset = (
        df_extended[df_extended["Air Pollutant"] == pollutant]
        .groupby("Year")["Air Pollution Level"]
        .median()
        .reset_index()
        .sort_values("Year")
    )

    ax.plot(
        subset["Year"],
        subset["Air Pollution Level"],
        marker="o",
        linewidth=2,
        color=colors_secondary[pollutant]
    )

    unit = (
        df_extended[df_extended["Air Pollutant"] == pollutant]
        ["Unit Of Air Pollution Level"]
        .dropna()
        .mode()
    )

    unit_text = unit.iloc[0] if len(unit) > 0 else ""

    ax.set_title(f"{pollutant} yearly median")
    ax.set_xlabel("Year")
    ax.set_ylabel(unit_text)
    ax.grid(alpha=0.3)

plt.suptitle("Trends of Secondary Pollutants (2015–2024)", fontsize=14)
plt.tight_layout()
plt.savefig("images/YoY_secondary_pollutans_median.png", dpi=300)
plt.show()

secondary_summary = (
    df_extended[df_extended["Air Pollutant"].isin(secondary_pollutants)]
    .groupby("Air Pollutant")["Air Pollution Level"]
    .median()
    .round(2)
)
secondary_summary.to_csv("images/YoY_secondary_pollutans_median.csv")
secondary_summary

Interpretation

Secondary pollutants show more diverse trends. CO and C6H6 display clear long-term declines, indicating improvements in combustion efficiency and emission controls. In contrast, SO₂ fluctuates, with a drop until 2018 followed by a gradual increase, suggesting localized industrial or energy-related influences rather than consistent policy-driven reductions.

## Conclusion

Key temporal findings:

- A visible drop in 2020 for traffic-related pollutants (NO2, PM) would confirm the COVID lockdown effect — strong evidence that human activity directly drives pollution.
- A consistent downward trend from 2015 onward would support the effectiveness of EU Clean Air policies.
- If pollution rebounded sharply in 2021-2022, that tells us the structural drivers (vehicles, industry) were not fundamentally changed by the pandemic.
- The 2022-2024 trend is the most policy-relevant: is progress continuing, or has it stalled?
- Secondary pollutants (SO₂, CO, C6H6) show relatively stable or gradually declining trends, suggesting that industrial combustion and fuel-related emissions have improved steadily over the last decade, though at a slower pace than traffic-related pollutants.


---
# EDA: Geographic Patterns

*Why this step?*

Air quality is deeply geographic. The same pollutant can vary enormously across countries, cities, and even within a city depending on where the monitoring station is located.

Three things drive this spatial variation:

- Country-level policy: some countries enforce stricter emission rules than others
- Station placement: a station next to a motorway always reads higher than one in a park
- Urban versus rural: cities concentrate vehicles and industry; rural areas typically have cleaner air

Spatial analysis tells us where the problem is worst, which determines where modeling efforts should focus.

## Bordering countries by median pollution level per pollutant

In [ ]:
# Comparison between Italy, Switzerland, Austria, France
combined = pd.concat([df_focus, df_compare_focus], ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, (pollutant, color) in enumerate(zip(focus_pollutants, colors)):
    sub = combined[combined['Air Pollutant'] == pollutant]
    country_median = (
        sub.groupby('Country')['Air Pollution Level']
        .median()
        .sort_values(ascending=False)
        .head(15)
    )
    country_median.plot(kind='bar', ax=axes[i], color=color, edgecolor='white')
    axes[i].set_title(f'{pollutant} — Top 15 Countries by Median Level')
    axes[i].set_ylabel('ug/m3')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Average Pollution by Country (Annual Median, 2015-2024)', fontsize=13)
plt.tight_layout()
plt.savefig("images/average_pollution_by_country.png", dpi=300)
plt.show()


Interpretation
Cross-Country Pollution Comparison (2015–2024)

- **Italy shows the highest pollution levels** across all primary pollutants:
  - Highest **NO₂, PM10, PM2.5**
  - Indicates strong impact from traffic, urban density, and geography (e.g., Po Valley)

- **Switzerland has the lowest pollution levels overall**
  - Lowest **PM10 and PM2.5**, low NO₂
  - Suggests effective environmental policies and better air circulation

- **France and Austria sit in the middle**
  - Moderate levels across pollutants
  - No extreme peaks → more balanced pollution profiles

- **O₃ behaves differently**
  - Similar levels across all countries
  - Less dependent on local emissions, more on atmospheric conditions


## Top polluted cities

In [ ]:
print("Unique cities in full df:", df["City"].nunique())
print("Unique cities in focus:", df_focus["City"].nunique())

In [ ]:
# Most Polluted Cities per Pollutant — Plotly with North/South colour annotation
# Latitude > 43 => North Italy, else South Italy

colors_map = {'NO2': 'steelblue', 'PM10': 'tomato', 'PM2.5': 'goldenrod', 'O3': 'mediumseagreen'}

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'{p} — Top 15 Most Polluted Cities' for p in focus_pollutants])

# Build a latitude lookup: median latitude per city
_lat_lookup = (
    df_focus[df_focus['City'].notna()]
    .groupby('City')['Latitude']
    .median()
)

for idx, pollutant in enumerate(focus_pollutants):
    row, col = divmod(idx, 2)
    sub = df_focus[
        (df_focus['Air Pollutant'] == pollutant) &
        (df_focus['City'].notna())
    ]
    city_stats = sub.groupby('City')['Air Pollution Level'].agg(['median', 'count'])
    city_stats = city_stats[city_stats['count'] >= 5]
    city_median = city_stats['median'].sort_values(ascending=False).head(15)

    # Assign North/South colour per city
    bar_colors = [
        '#4c72b0' if _lat_lookup.get(city, 43) > 43 else '#dd8452'
        for city in city_median.index
    ]

    fig.add_trace(
        go.Bar(
            y=city_median.index.tolist(),
            x=city_median.values,
            orientation='h',
            marker_color=bar_colors,
            name=pollutant,
            showlegend=False
        ),
        row=row+1, col=col+1
    )

# Add dummy traces for legend (North / South)
for label, color in [('North (lat > 43°)', '#4c72b0'), ('South (lat ≤ 43°)', '#dd8452')]:
    fig.add_trace(go.Bar(x=[None], y=[None], orientation='h',
                         marker_color=color, name=label, showlegend=True))

fig.update_layout(
    title_text='Most Polluted Cities per Pollutant (Annual Median, 2015-2024) — Blue=North, Orange=South',
    height=900, width=1200,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(title='Italian macro-area')
)
fig.update_xaxes(title_text='ug/m3', gridcolor='#CCCCCC')
fig.update_yaxes(autorange='reversed', gridcolor='#CCCCCC')
fig.write_image('images/Most polluted cities per polluntant.png', scale=2)
fig.show()

Interpretation
City-Level Pollution

- **North dominates primary pollution (NO₂, PM10, PM2.5)**
  - Cities like Milano, Bergamo, Cremona appear repeatedly
  - Driven by traffic, industry, and Po Valley effects

- **South appears less in primary pollutants**
  - Indicates lower urban/industrial pressure

- **O₃ shows the opposite pattern**
  - Southern cities dominate → linked to heat and sunlight

- **North = human-driven pollution**
- **South = climate-driven pollution (O₃)**

## Dynamic city-level map

To better understand spatial and temporal pollution dynamics, we aggregate station-level measurements to the city–year level. For each city and year, pollutant concentrations are computed as the median across available monitoring stations.

This aggregated dataset is then enriched with external population data, allowing us to visualise how pollution patterns evolve across Italian cities over time.

#### Map 1: Limit‑weighted pollution index based on city/year/pollutant distribution

Colour = mean over pollutants of (concentration / EU limit). Values &gt; 1 mean the city–year exceeds at least one limit on average. Circle size = population. Years 2015–2024.

In [ ]:
# ═══════════════════════════════════════════════════════
# EEA 2024 official AQI breakpoints & helper constants
# ═══════════════════════════════════════════════════════

EU_LIMITS  = {"PM2.5": 25, "PM10": 40, "NO2": 40, "O3": 120}
WHO_GUIDE  = {"PM2.5": 5,  "PM10": 15, "NO2": 10, "O3": 60}

EEA_BANDS = {
    "PM2.5": [5, 15, 50, 90, 140],
    "PM10":  [15, 45, 120, 195, 270],
    "NO2":   [10, 25, 60, 100, 150],
    "O3":    [60, 100, 120, 160, 180],
}

BAND_NAMES  = ["Good", "Fair", "Moderate", "Poor", "Very poor", "Extremely poor"]
BAND_COLORS = ["#50F0E6", "#50CCAA", "#F0E641", "#FF5050", "#960032", "#7D2181"]

def eea_band(pollutant, conc):
    """Return 1-6 band index for a concentration value."""
    if pd.isna(conc):
        return np.nan
    thresholds = EEA_BANDS.get(pollutant)
    if thresholds is None:
        return np.nan
    for i, t in enumerate(thresholds):
        if conc <= t:
            return i + 1
    return 6

def overall_aqi(row, pollutants):
    """Overall AQI = worst (max) sub-index among available pollutants."""
    vals = [eea_band(p, row.get(p, np.nan)) for p in pollutants]
    vals = [v for v in vals if not pd.isna(v)]
    return max(vals) if vals else np.nan

def dominant_pollutant(row, pollutants):
    """Return the name of the pollutant that drives the AQI."""
    worst_band = -1
    worst_pol  = np.nan
    for p in pollutants:
        b = eea_band(p, row.get(p, np.nan))
        if not pd.isna(b) and b > worst_band:
            worst_band = b
            worst_pol  = p
    return worst_pol

print("EEA 2024 helpers loaded ✓")

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "browser"

map1_df = merged.copy()

for p in pol_cols:
    lim = EU_LIMITS.get(p)
    if lim and p in map1_df.columns:
        map1_df[f"ratio_{p}"] = map1_df[p] / lim

ratio_cols = [c for c in map1_df.columns if c.startswith("ratio_")]
map1_df["Weighted_index"] = map1_df[ratio_cols].mean(axis=1)

map1_df = map1_df.dropna(subset=["Latitude", "Longitude", "Weighted_index", "City_Population"])

fig1 = px.scatter_map(
    map1_df,
    lat="Latitude",
    lon="Longitude",
    size="City_Population",
    color="Weighted_index",
    animation_frame="Year",
    animation_group="City",
    hover_data=["City", "City_Population"] + pol_cols,
    map_style="open-street-map",
    zoom=5,
    center=dict(lat=42.8, lon=12.5),
    title="Map 1: Limit-weighted pollution index (2015–2024)",
    size_max=40,
    range_color=(0, 1)
)

fig1.update_layout(
    template="plotly_white",
    height=900,
    width=1200,
    margin={"r": 0, "l": 0, "t": 50, "b": 0},
    coloraxis_colorbar_title="Weighted index<br>(mean conc/limit)",
    annotations=[
        dict(
            x=0.02, y=0.98, xref="paper", yref="paper",
            text="<b>Map 1.</b> Colour = limit-weighted index. Size = population. Index > 1 = above EU limits on average.",
            showarrow=False,
            font=dict(size=11),
            align="left",
            bgcolor="rgba(255,255,255,0.8)",
            borderpad=6,
        )
    ],
)

fig1.show()
pio.renderers.default = "notebook"  # restore netebook render for remaining cells
print("Map is shown in browser")

Interpretation
Limit-Weighted Pollution Index (City Level)

- **Clear spatial clustering in the North (Po Valley)**
  - Highest index values concentrated in Northern Italy
  - Indicates frequent exceedance of EU limits

- **Large cities stand out**
  - Bigger bubbles (e.g., Milano, Roma) -> high population + pollution impact

- **South shows lower index values overall**
  - Fewer exceedances of EU limits
  - More dispersed and less intense pollution

- **Temporal consistency**
  - Patterns remain stable across years → structural issue, not temporary

- **Pollution in Italy is spatially concentrated and persistent**
- **North = high-risk zone (systemic exceedance of limits)**

## Station type and area comparison

In [ ]:
# Station Context — Area only — Plotly grouped bar
area_median = (
    df_focus.groupby(['Air Quality Station Area', 'Air Pollutant'])['Air Pollution Level']
    .median().reset_index()
)
area_pivot = area_median.pivot(index='Air Quality Station Area',
                                columns='Air Pollutant',
                                values='Air Pollution Level')[focus_pollutants]

fig = go.Figure()
pollutant_colors = {'NO2': '#4c72b0', 'PM10': '#55a868', 'PM2.5': '#c44e52', 'O3': '#8172b2'}

for p in focus_pollutants:
    fig.add_trace(
        go.Bar(name=p, x=area_pivot.index.tolist(), y=area_pivot[p].values,
               marker_color=pollutant_colors[p])
    )

fig.update_layout(
    title_text='Median Level by Station Area',
    barmode='group',
    height=500, width=700,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(title='Pollutant'),
    xaxis_title='Station Area',
    yaxis_title='ug/m3',
)
fig.update_xaxes(tickangle=-30, gridcolor='#CCCCCC')
fig.update_yaxes(gridcolor='#CCCCCC')
fig.write_image('images/station_context.png', scale=2)
fig.show()

Interpretation
Station Area

Higher pollution levels are clearly associated with more urbanized areas. Urban stations show the highest concentrations of NO₂ and particulate matter, reflecting traffic and human activity intensity.

- Urban → highest NO₂, PM10, PM2.5
- Suburban → moderate levels
- Rural → lowest primary pollutants

Ozone behaves differently, with higher levels observed in rural areas due to atmospheric formation rather than direct emissions.

In [ ]:
# Check dependencies
print("df_focus" in dir() and df_focus is not None)  # should be True
print("focus_pollutants:", focus_pollutants)

## Secondary Pollutants by Station Context

In [ ]:
# Secondary pollutants by station context
secondary_subset = df_extended[
    df_extended["Air Pollutant"].isin(secondary_pollutants)
].copy()

# Median pollution by station type
type_median_ext = (
    secondary_subset
    .groupby(["Air Quality Station Type", "Air Pollutant"])["Air Pollution Level"]
    .median()
    .reset_index()
)

type_pivot_ext = type_median_ext.pivot(
    index="Air Quality Station Type",
    columns="Air Pollutant",
    values="Air Pollution Level"
)

# Median pollution by station area
area_median_ext = (
    secondary_subset
    .groupby(["Air Quality Station Area", "Air Pollutant"])["Air Pollution Level"]
    .median()
    .reset_index()
)

area_pivot_ext = area_median_ext.pivot(
    index="Air Quality Station Area",
    columns="Air Pollutant",
    values="Air Pollution Level"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot by station type
type_pivot_ext[secondary_pollutants].plot(
    kind="bar",
    ax=axes[0],
    edgecolor="black"
)
axes[0].set_title("Secondary Pollutants by Station Type")
axes[0].set_ylabel("Median Pollution Level")
axes[0].set_xlabel("Station Type")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

# Plot by station area
area_pivot_ext[secondary_pollutants].plot(
    kind="bar",
    ax=axes[1],
    edgecolor="black"
)
axes[1].set_title("Secondary Pollutants by Station Area")
axes[1].set_ylabel("Median Pollution Level")
axes[1].set_xlabel("Station Area")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("images/secondary_by_station.png", dpi=300)
plt.show()

**Note: CO concentrations are measured in mg/m³ while SO₂ and C6H6 are measured in µg/m³. The comparison focuses on relative differences across station contexts rather than absolute magnitudes. Graph is meant to be observed as comparison of one pollutant across different environments, rather than comparison of between pollutants**

## Conclusion

Key spatial findings:

- Pollution is not evenly distributed. A small number of countries and cities drive the European average significantly upward.
- Traffic stations consistently record higher NO2 than Background stations. This directly reflects vehicle emissions and validates the monitoring approach.
- Urban areas show systematically higher levels than suburban or rural ones. This is the scientific basis for urban clean air zones.
- The city ranking gives us a hotspot map of where the air quality problem is concentrated — these cities should be the priority for both policy action and model validation.
- Secondary pollutants vary across monitoring contexts. Industrial stations show the highest SO₂ levels, while traffic and urban areas display higher CO and C6H6 concentrations. These patterns support their links to industrial combustion and vehicle emissions, helping identify potential emission sources for future modeling.




## Exploring the North–South Pollution Gradient in Italy

In [ ]:
# Italy-only subset with valid coordinates
df_it_focus = df_focus[
    (df_focus["Country"] == "Italy") &
    (df_focus["Latitude"].notna()) &
    (df_focus["Longitude"].notna()) &
    (df_focus["Air Pollution Level"].notna())
].copy()

print("Rows used for Italy North-South comparison:", f"{len(df_it_focus):,}")
print("Pollutants included:", df_it_focus["Air Pollutant"].nunique())
print(df_it_focus["Air Pollutant"].value_counts())

In [ ]:
# Split Italy into macro-areas using a fixed geographic threshold
df_it_focus["Italy_area"] = np.where(
    df_it_focus["Latitude"] > 43,
    "North",
    "South"
)

display(
    df_it_focus[
        ["City", "Latitude", "Air Pollutant", "Air Pollution Level", "Italy_area"]
    ].head(10)
)

In [ ]:
# Build a comparable composite pollution measure
# Raw pollutant levels cannot be averaged directly because pollutants have different scales.
# We therefore normalize each pollutant by its EU reference limit.

EU_LIMITS = {
    "PM10": 40,
    "PM2.5": 25,
    "NO2": 40,
    "O3": 120
}

df_it_focus["Pollution_ratio"] = (
    df_it_focus["Air Pollution Level"] /
    df_it_focus["Air Pollutant"].map(EU_LIMITS)
)

# Cap very extreme normalized values to reduce distortion (Maybe will delete it)
cap_ratio = df_it_focus["Pollution_ratio"].quantile(0.999)
df_it_focus["Pollution_ratio_capped"] = df_it_focus["Pollution_ratio"].clip(upper=cap_ratio)

print(f"99.9th percentile cap for normalized ratio: {cap_ratio:.2f}")

In [ ]:
# Summary statistics for North vs South
north_south_index = (
    df_it_focus
    .groupby("Italy_area")["Pollution_ratio_capped"]
    .agg(["count", "mean", "median", "std"])
    .round(3)
    .sort_values("mean", ascending=False)
)

display(north_south_index)
north_south_index.to_csv("images/italy_north_south_composite_index.csv")

In [ ]:
# Visual comparison of normalized pollution
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df_it_focus,
    x="Italy_area",
    y="Pollution_ratio_capped",
    hue="Italy_area",
    legend=False,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 4},
    palette=["#4c72b0", "#dd8452"]
)

plt.title("North vs South Italy — Composite Pollution Index")
plt.xlabel("Italian macro-area")
plt.ylabel("Pollution / EU limit")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("images/north_south_composite_pollution.png", dpi=300)
plt.show()

In [ ]:
# Pollutant-level comparison for transparency
north_south_by_pollutant = (
    df_it_focus
    .groupby(["Air Pollutant", "Italy_area"])["Air Pollution Level"]
    .median()
    .unstack()
    .reindex(focus_pollutants)
    .round(2)
)

display(north_south_by_pollutant)
north_south_by_pollutant.to_csv("images/north_south_pollutant_medians.csv")

In [ ]:
# Clear textual output
north_mean = north_south_index.loc["North", "mean"]
south_mean = north_south_index.loc["South", "mean"]
north_median = north_south_index.loc["North", "median"]
south_median = north_south_index.loc["South", "median"]

print(f"Mean composite pollution index (North): {north_mean:.3f}")
print(f"Mean composite pollution index (South): {south_mean:.3f}")
print(f"Median composite pollution index (North): {north_median:.3f}")
print(f"Median composite pollution index (South): {south_median:.3f}")

if north_median > south_median:
    print("\nNorthern Italy appears more polluted overall.")
elif south_median > north_median:
    print("\nSouthern Italy appears more polluted overall.")
else:
    print("\nNorth and South show very similar overall pollution levels.")

Interpretation

North–South Pollution Gradient

Northern Italy shows slightly higher overall pollution levels, both in mean and median composite index. The difference is not extreme but consistent across the dataset.

- Higher in North → NO₂, PM10, PM2.5
- Higher in South → O₃

This reflects different drivers: human activity and geography in the North vs atmospheric conditions in the South.

## Extending the Spatial Analysis: Environmental Drivers

The geographic exploration revealed that air pollution is not randomly distributed across space.
Instead, clear spatial patterns emerge across countries, cities, and monitoring station contexts.

Several results from the previous analysis motivate a deeper investigation into environmental variables:

- **Country comparisons** showed systematic differences in pollutant levels across neighboring countries, suggesting that geographic and environmental factors may influence air quality beyond local emissions.
- **City-level rankings** revealed persistent pollution hotspots, many of which are located in dense urban or industrial areas.
- **Station context analysis** demonstrated that traffic and urban stations consistently report higher pollutant concentrations than rural or background stations.

These findings indicate that **urban structure and geographic characteristics** likely play an important role in explaining pollution variability.

### Hypothesis: Environmental Context Matters

Two environmental variables are particularly relevant for understanding spatial differences in pollution:

- **Altitude**
  Higher Altitude typically experience stronger wind circulation and atmospheric mixing, which can help disperse pollutants.
  In contrast, low-altitude areas—especially valleys or basins—can trap polluted air and lead to higher concentrations.

- **Green Space Ratio**
  Vegetation can influence air quality by:
  - capturing particulate matter on leaf surfaces
  - promoting airflow and cooling effects
  - reducing urban heat island intensity

Cities with higher proportions of parks, forests, or vegetated land may therefore experience lower pollution levels.

### Motivation from the North–South Comparison

A preliminary comparison between **Northern and Southern regions** shows that northern areas tend to exhibit higher pollution levels.

This pattern is consistent with known environmental conditions in Italy and central Europe:

- Northern regions contain **large industrial and urban clusters**
- The **Po Valley**, in particular, is characterized by **low elevation and limited atmospheric circulation**
- Southern regions generally have **higher terrain variation and more open landscapes**

These observations suggest that **topography and land cover** may partially explain the spatial variation observed in the dataset.

### Next Step

To explore these hypotheses, we incorporate **Green space ratio** around stations or municipalities

These variables will allow us to test whether environmental context helps explain pollution differences across regions and cities, and whether they improve predictive modeling in later stages of the project.

# Features of Interest

## Exploring Altitude Data

In [ ]:
# Build PM2.5 dataframe using station altitude
df_pm25_alt = df_focus[
    (df_focus["Air Pollutant"] == "PM2.5") &
    (df_focus["Altitude"].notna())
    ].copy()

print("PM2.5 rows with station altitude:", f"{len(df_pm25_alt):,}")
print("Unique stations:", df_pm25_alt["Air Quality Station EoI Code"].nunique())
print("Years covered:", sorted(df_pm25_alt["Year"].dropna().unique().tolist()))

display(
    df_pm25_alt[
        ["City", "Year", "Air Pollution Level", "Altitude"]
    ].head(10)
)

In [ ]:
# PM2.5 vs Station Altitude — OLS (drop NaNs so slope/intercept are not nan)
import numpy as np
import plotly.graph_objects as go

x_vals = df_pm25_alt['Altitude'].values.astype(float)
y_vals = df_pm25_alt['Air Pollution Level'].values.astype(float)

# Drop rows where either Altitude or Air Pollution Level is NaN
mask = np.isfinite(x_vals) & np.isfinite(y_vals)
x_vals = x_vals[mask]
y_vals = y_vals[mask]
n = len(x_vals)

if n < 2:
    slope = 0.0
    intercept = np.nanmean(df_pm25_alt['Air Pollution Level']) if n == 1 else 0.0
    print("Warning: Not enough valid points for regression.")
else:
    # OLS
    denom = n * np.sum(x_vals**2) - np.sum(x_vals)**2
    if np.abs(denom) < 1e-14:
        slope = 0.0
        intercept = np.mean(y_vals)
        print("Warning: Altitude has no variance; line is horizontal.")
    else:
        slope = (n * np.sum(x_vals * y_vals) - np.sum(x_vals) * np.sum(y_vals)) / denom
        intercept = np.mean(y_vals) - slope * np.mean(x_vals)

print(f"Slope = {slope:.6f}, Intercept = {intercept:.4f}, n = {n}")

x_min = x_vals.min()
x_max = x_vals.max()
x_line = np.linspace(x_min, x_max, 200)
y_line = slope * x_line + intercept

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_pm25_alt['Altitude'], y=df_pm25_alt['Air Pollution Level'],
    mode='markers', marker=dict(size=5, color='steelblue', opacity=0.6),
    name='Stations'
))
fig.add_trace(go.Scatter(
    x=x_line, y=y_line, mode='lines',
    line=dict(color='red', width=3),
    name=f'OLS (slope={slope:.4f})'
))
fig.update_layout(
    title='PM2.5 vs Station Altitude',
    xaxis_title='Altitude (m)', yaxis_title='PM2.5 (ug/m3)',
    height=450, width=800,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(gridcolor='#CCCCCC'), yaxis=dict(gridcolor='#CCCCCC')
)
fig.write_image('images/pm25_vs_station_altitude.png', scale=2)
fig.show(renderer='notebook')

In [ ]:
# Correlation comparison: station altitude vs municipal elevation

corr_pm25_alt = df_pm25_alt[["Air Pollution Level", "Altitude"]].corr().iloc[0, 1]

print(f"Correlation between PM2.5 and station altitude: {corr_pm25_alt:.3f}")


In [ ]:
# Elevation quartiles using station altitude
df_pm25_alt["altitude_band"] = pd.qcut(
    df_pm25_alt["Altitude"],
    q=4,
    labels=["Q1: Lowest", "Q2", "Q3", "Q4: Highest"]
)

display(
    df_pm25_alt[["City", "Altitude", "altitude_band"]]
    .drop_duplicates()
    .head(10)
)

# Boxplot PM2.5 by altitude band
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df_pm25_alt,
    x="altitude_band",
    y="Air Pollution Level",
    hue="altitude_band",
    legend=False,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 4},
    palette="Blues"
)

plt.title("PM2.5 Distribution by Station Altitude Quartile")
plt.xlabel("Altitude quartile")
plt.ylabel("PM2.5 annual median (µg/m³)")
plt.tight_layout()
plt.savefig("images/pm25_by_station_altitude_quartile.png", dpi=300)
plt.show()

## Interpretation

The correlation between PM2.5 levels and station altitude is −0.315, indicating a moderate negative relationship. Stations located at higher altitudes tend to record lower PM2.5 concentrations on average.

This pattern is also visible in the data: the regression line shows a clear downward trend, and PM2.5 levels decrease across higher altitude quartiles.

This relationship is consistent with known atmospheric dynamics. Low altitude areas particularly valleys and basins are more prone to pollutant accumulation due to weaker air circulation and temperature inversions. In contrast, higher-altitude locations generally experience stronger atmospheric mixing and wind dispersion, which likely contributes to lower particulate concentrations.

## Calculating Population Density

In [ ]:
# comuni area lookup
_comuni_area = comuni[["COMUNE", "Shape_Area"]].copy()
_comuni_area["City_norm"] = _comuni_area["COMUNE"].apply(normalize_city_for_merge)
_comuni_area["City_Area_km2"] = (_comuni_area["Shape_Area"] / 1e6).round(1)

# UN primary
_latest = int(un_pop_filled["Year"].max())
_un = (
    un_pop_filled[un_pop_filled["Year"] == _latest]
    [["City", "City_norm", "Population_UN"]]
    .drop_duplicates(subset="City_norm")
    .rename(columns={"Population_UN": "City_Population"})
    .merge(_comuni_area[["City_norm", "City_Area_km2"]], on="City_norm", how="left")
    .dropna(subset=["City_Area_km2"])
)
_un["Pop_Density"] = (_un["City_Population"] / _un["City_Area_km2"]).round(0)

# ISTAT fallback — only cities NOT in UN
_un_cities = set(_un["City_norm"])
_istat = (
    istat_pop[~istat_pop["City_norm"].isin(_un_cities)]
    .merge(_comuni_area[["City_norm", "City_Area_km2"]], on="City_norm", how="left")
    .dropna(subset=["City_Area_km2"])
    .drop_duplicates(subset="City_norm")
    .assign(
        City=lambda d: d["City_ISTAT"],
        City_Population=lambda d: d["Population_ISTAT"],
        Pop_Density=lambda d: (d["Population_ISTAT"] / d["City_Area_km2"]).round(0)
    )
    [["City", "City_norm", "City_Population", "City_Area_km2", "Pop_Density"]]
)

# Combine
pop_density_cities = (
    pd.concat([_un, _istat], ignore_index=True)
    .drop_duplicates(subset="City_norm")
    [["City", "City_norm", "City_Population", "City_Area_km2", "Pop_Density"]]
)

# Add EEA-style aliases for (greater city) variants
_eea_cities = (
    df[["City"]].dropna().drop_duplicates().copy()
)
_eea_cities["City_norm"] = (
    _eea_cities["City"].str.strip().str.lower()
    .str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()
)
_eea_aliases = (
    _eea_cities
    .merge(
        pop_density_cities[["City_norm", "City_Population", "City_Area_km2", "Pop_Density"]],
        on="City_norm", how="inner"
    )
)
pop_density_cities = (
    pd.concat([pop_density_cities, _eea_aliases], ignore_index=True)
    .drop_duplicates(subset="City")
    [["City", "City_norm", "City_Population", "City_Area_km2", "Pop_Density"]]
)

print(f"UN matched:           {len(_un)}")
print(f"ISTAT fallback added: {len(_istat)}")
print(f"Total after aliases:  {len(pop_density_cities)}")
display(
    pop_density_cities[["City", "City_Population", "City_Area_km2", "Pop_Density"]]
    .sort_values("Pop_Density", ascending=False)
    .reset_index(drop=True)
)

### Population density and exposure

We need city area (from comuni boundaries) to compute population density and density-weighted exposure. If `merged` already has `City_Area_km2`, `Pop_Density`, and `Exposure_Dense` from the maps section, you can skip the next cell.

In [ ]:
# Ensure merged has City_Area_km2, Pop_Density, Limit_Index, Exposure_Dense (for density & exposure analysis)
import geopandas as gpd
import numpy as np

# Comune -> City mapping from stations
_st2c = (
    joined.dropna(subset=["COMUNE"])
    .groupby("Air Quality Station EoI Code")["COMUNE"]
    .agg(lambda s: s.mode().iloc[0])
)
df_map = df_focus.copy()
df_map["Comune"] = df_map["Air Quality Station EoI Code"].map(_st2c)
_comune_to_city = (
    df_map.dropna(subset=["Comune", "City"])
    .groupby("Comune")["City"]
    .agg(lambda s: s.mode().iloc[0])
    .apply(lambda s: str(s).strip().lower().title())
)
_comuni_wc = comuni.copy()
_comuni_wc["City"] = _comuni_wc["COMUNE"].map(_comune_to_city)
_comuni_wc = _comuni_wc.dropna(subset=["City"])
city_geo = _comuni_wc.dissolve(by="City").reset_index()[["City", "geometry"]]
city_geo = city_geo.to_crs("EPSG:4326")

# City area (km²) and density
_city_ea = city_geo.to_crs("EPSG:3035")
_city_ea["area_km2"] = _city_ea.geometry.area / 1e6
merged["City_Area_km2"] = merged["City"].map(_city_ea.set_index("City")["area_km2"])
merged["Pop_Density"] = merged["City_Population"] / merged["City_Area_km2"]

# Limit index and exposure (if not already from maps setup)
pol_cols = [p for p in focus_pollutants if p in merged.columns]
if "Limit_Index" not in merged.columns:
    for p in pol_cols:
        lim = EU_LIMITS.get(p)
        if lim and p in merged.columns:
            merged[f"ratio_{p}"] = merged[p] / lim
    _mr = [c for c in merged.columns if c.startswith("ratio_")]
    merged["Limit_Index"] = merged[_mr].mean(axis=1)
merged["Exposure_Dense"] = merged["Pop_Density"] * merged["Limit_Index"]

print("Merged now has City_Area_km2, Pop_Density, Limit_Index, Exposure_Dense")

In [ ]:
print("joined" in dir())
print(len(joined) if "joined" in dir() else "N/A")

#### Density vs pollution (city-level, median 2015–2024)

Each dot is one city (median Pop_Density and median concentration over years). Red dashed line = linear fit; label = Pearson r. Orange/red horizontal lines = WHO and EU thresholds. If density and concentration are positively correlated, urbanisation is a driver of exposure.

In [ ]:
import matplotlib.pyplot as plt

city_avg = (
    merged.dropna(subset=["Pop_Density"])
    .groupby("City")
    .agg({"Pop_Density": "median", **{p: "median" for p in pol_cols}})
)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
for i, p in enumerate(pol_cols):
    ax = axes[i]
    data = city_avg.dropna(subset=[p, "Pop_Density"])
    x, y = data["Pop_Density"].values, data[p].values
    ax.scatter(x, y, alpha=0.55, s=55, color="steelblue", edgecolors="white", lw=0.5)
    if len(data) > 3:
        m, b = np.polyfit(x, y, 1)
        r = np.corrcoef(x, y)[0, 1]
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, m * xs + b, "r--", alpha=0.7, label=f"r = {r:.2f}")
    if p in WHO_GUIDE:
        ax.axhline(WHO_GUIDE[p], color="orange", ls=":", lw=1.5, label=f"WHO ({WHO_GUIDE[p]})")
    if p in EU_LIMITS:
        ax.axhline(EU_LIMITS[p], color="red", ls=":", lw=1.5, label=f"EU ({EU_LIMITS[p]})")
    ax.set_xlabel("Population density (people / km²)")
    ax.set_ylabel(f"{p} (µg/m³)")
    ax.set_title(p, fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("images/density_vs_pollution.png", dpi=150, bbox_inches="tight")
plt.show()

###  Interpretation Pollution Concentration vs. Population Density

These scatter plots analyze the correlation between **Population Density** ($people/km^2$) and the concentrations of four key pollutants (NO2, PM10, PM2.5, and O3) across Italian cities.

---

Pollutant-Specific Trends

1. Nitrogen Dioxide (NO2)
* **Strongest Correlation ($r = 0.57$):** NO2 shows a clear positive linear relationship with density. Since NO2 is primarily a byproduct of road traffic and combustion, higher urban density (and associated traffic) directly increases concentrations.
* **Limit Status:** While most cities stay below the EU limit (40), almost **all cities exceed the WHO guideline (10)**.

2. Particulate Matter (PM10 & PM2.5)
* **Weak to Moderate Correlation ($r = 0.18$ to $0.31$):** Particulates show a positive but much "noisier" relationship with density.
* **Factors:** PM levels are influenced not only by local traffic but also by industrial activity and geographic factors (like the Po Valley's tendency to trap particles), which can affect low and high-density cities alike.
* **Limit Status:** Nearly all cities exceed WHO guidelines for both PM10 and PM2.5, though most remain within EU regulatory limits.

3. Ozone (O3)
* **Negative Correlation ($r = -0.19$):** Interestingly, Ozone shows a slight inverse relationship with population density.
* **Chemical Titration:** This is a known phenomenon in urban chemistry; in very dense areas with high NO emissions (from traffic), the NO reacts with and "consumes" Ozone locally. Consequently, Ozone levels are often higher in suburban or rural areas than in dense city centers.
* **Limit Status:** Most cities hover around or exceed the WHO guideline (60), but stay well below the EU limit (120).


#### Population-weighted mean concentration (Italy, 2015–2024)

For each year, the mean concentration is weighted by city population so that large cities count more. Dotted lines = WHO guidelines. Shows how far the nationally weighted average is from health targets.

In [ ]:
import plotly.graph_objects as go

POL_COLORS = {"NO2": "#4363d8", "PM10": "#f58231", "PM2.5": "#e6194B", "O3": "#3cb44b"}

records = []
for year in sorted(merged["Year"].unique()):
    sub = merged[merged["Year"] == year].dropna(subset=["City_Population"])
    if sub["City_Population"].sum() == 0:
        continue
    row = {"Year": int(year)}
    for p in pol_cols:
        v = sub.dropna(subset=[p])
        if len(v) > 0:
            row[p] = (v[p] * v["City_Population"]).sum() / v["City_Population"].sum()
    records.append(row)
pw_trend = pd.DataFrame(records)

fig_d = go.Figure()
for p in pol_cols:
    if p not in pw_trend.columns:
        continue
    fig_d.add_trace(go.Scatter(
        x=pw_trend["Year"], y=pw_trend[p],
        mode="lines+markers", name=p,
        line=dict(color=POL_COLORS.get(p, "gray"), width=2.5),
        marker=dict(size=8),
    ))
    if p in WHO_GUIDE:
        fig_d.add_hline(y=WHO_GUIDE[p], line_dash="dot",
                        annotation_text=f"WHO {p}",
                        line_color=POL_COLORS.get(p, "gray"), opacity=0.5)

fig_d.update_layout(
    title="Population-weighted mean concentration (µg/m³), Italy 2015–2024",
    xaxis_title="Year", yaxis_title="µg/m³",
    height=550, width=1000, template="plotly_white",
    legend=dict(x=0.01, y=0.99),
)
fig_d.show()

S### Interpretation: Population-weighted Mean Concentration Trends (2015–2024)

This time-series chart tracks the average exposure of the Italian population to major pollutants, providing a macro-view of air quality progress relative to **WHO Guidelines** (dashed lines).

**Trends by Pollutant:**

* **NO2 (Blue):** Shows the most significant improvement, dropping from nearly **30 $ug/m^3$** in 2015 to approximately **20 $ug/m^3$** by 2024. Despite this decline, it remains consistently above the **WHO NO2** limit (10).
* **PM10 (Orange) & PM2.5 (Red):** Both particulates show a steady, gradual decline.
    * **PM10** has converged with NO2 levels recently (~20 $ug/m^3$).
    * **PM2.5** remains the most "compliant" relative to its 2015 starting point but still sits well above the very strict **WHO PM2.5** limit (5).
* **O3 (Green):** Unlike the others, Ozone concentrations have remained high and relatively flat, hovering between **50 and 60 $ug/m^3$**. It is the only pollutant that consistently approaches or stays near its **WHO O3** threshold throughout the decade.

**Observations:**

* **The 2020 "Dip":** There is a visible sharp decline in **NO2** in 2020, likely attributable to reduced traffic and industrial activity during COVID-19 lockdowns.
* **Decoupling:** While NO2 and Particulates are trending downward (indicating successful emission controls for vehicles and industry), **Ozone** persists as a challenging secondary pollutant that does not follow the same downward trajectory.
* **The WHO Gap:** Across all pollutants, the population-weighted mean remains above WHO health recommendations as of 2024, highlighting a continued public health challenge despite the general downward trend in primary pollutants.

## Integrating Green Space Ratio

In [ ]:
# ── CORINE Green Space Ratio ──
df_green = pd.read_csv("LandCover2018/green_ratio.csv")

# Create Italy dataframe first
df_focus_it_geo = df_focus[df_focus["Country"] == "Italy"].copy()

df_focus_it_geo = df_focus_it_geo.merge(
    df_green[["Air Quality Station EoI Code", "green_ratio"]],
    on="Air Quality Station EoI Code",
    how="left"
)

print("Rows after merge:", len(df_focus_it_geo))
print("Missing green_ratio:", df_focus_it_geo["green_ratio"].isna().sum())
df_focus_it_geo[["Air Quality Station EoI Code", "green_ratio"]].head(5)

In [ ]:
# Correlation
corr = (
    df_focus_it_geo[df_focus_it_geo["Air Pollutant"] == "PM2.5"]
    [["Air Pollution Level", "green_ratio"]]
    .corr().iloc[0, 1]
)
print(f"Correlation between green_ratio and PM2.5: {corr:.3f}")
print("Negative = greener comuni tend to have lower PM2.5 — expected direction")

In [ ]:
# Green Ratio vs PM2.5: scatter + boxplot — Plotly
# (named 'population density and human activity' in the chart list)

pm25_per_station = (
    df_focus_it_geo[df_focus_it_geo['Air Pollutant'] == 'PM2.5']
    .groupby('Air Quality Station EoI Code')['Air Pollution Level']
    .median().reset_index()
    .rename(columns={'Air Pollution Level': 'pm25_median'})
)

plot_df = pm25_per_station.merge(
    df_focus_it_geo[['Air Quality Station EoI Code', 'green_ratio']].drop_duplicates(),
    on='Air Quality Station EoI Code', how='inner'
).dropna()

plot_df['green_band'] = pd.qcut(
    plot_df['green_ratio'], q=4,
    labels=['Q1 Most urban', 'Q2', 'Q3', 'Q4 Most green']
)

print(f'Stations used: {len(plot_df)}')

import numpy as np
from scipy import stats as _stats

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=[
                        'PM2.5 vs Green Space Ratio (per station, Italy 2015–2024)',
                        'PM2.5 by Green Space Quartile (per station, Italy 2015–2024)'
                    ])

# Left: scatter with OLS regression line
fig.add_trace(
    go.Scatter(x=plot_df['green_ratio'], y=plot_df['pm25_median'],
               mode='markers',
               marker=dict(color='steelblue', opacity=0.4, size=6),
               name='Station', showlegend=False),
    row=1, col=1
)
slope, intercept, *_ = _stats.linregress(plot_df['green_ratio'], plot_df['pm25_median'])
x_line = np.linspace(plot_df['green_ratio'].min(), plot_df['green_ratio'].max(), 100)
fig.add_trace(
    go.Scatter(x=x_line, y=slope*x_line + intercept,
               mode='lines', line=dict(color='red', width=2),
               name='OLS fit', showlegend=False),
    row=1, col=1
)

# Right: boxplot by quartile
band_palette = {'Q1 Most urban': '#e05c5c', 'Q2': '#f5a623',
                'Q3': '#a8d5a2', 'Q4 Most green': '#4caf50'}
for band in ['Q1 Most urban', 'Q2', 'Q3', 'Q4 Most green']:
    vals = plot_df[plot_df['green_band'] == band]['pm25_median']
    fig.add_trace(
        go.Box(y=vals, name=band, marker_color=band_palette[band],
               boxpoints='outliers', showlegend=False),
        row=1, col=2
    )

fig.update_layout(
    title_text='Green Space vs PM2.5',
    height=500, width=1200,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)'
)
fig.update_xaxes(gridcolor='#CCCCCC')
fig.update_yaxes(gridcolor='#CCCCCC')
fig.update_xaxes(title_text='Green Ratio (0=fully urban, 1=fully green)', row=1, col=1)
fig.update_yaxes(title_text='Median PM2.5 (µg/m³)', row=1, col=1)
fig.update_xaxes(title_text='Green Space Band', row=1, col=2)
fig.update_yaxes(title_text='Median PM2.5 (µg/m³)', row=1, col=2)
fig.write_image('images/population density and human activity.png', scale=2)
fig.show()

corr = plot_df[['green_ratio', 'pm25_median']].corr().iloc[0, 1]
print(f'\nCorrelation green_ratio vs PM2.5: {corr:.3f}')

**Does the greenness of a municipality explain the pollution level of the station located there?**
Green space ratio is a static property of a place — it does not change year to year in our data. Correlating it against 10 repeated
annual measurements per station would artificially inflate the sample size and give each station unequal weight.

By taking the **median PM2.5 across all years per station**, we get one representative value per location — which is the right unit of comparison when the predictor variable (green_ratio) is also measured at the station level.

## df_model is built in features.ipynb

Based on the EDA findings above (altitude, green ratio, population density, station type/area all show meaningful relationships with PM2.5), we selected these as features.

The full feature matrix is constructed in features.ipynb and saved as df_model.csv.



In [ ]:
df_model = pd.read_csv('df_model.csv')

print('df_model shape:', df_model.shape)
print('\nMissing values:')
print(df_model.isnull().sum())
df_model.head()

### Interpretation (Features of Interest)

Green Space and PM2.5

A moderate negative relationship is observed between green space ratio and PM2.5 levels (correlation ≈ −0.295). Stations located in greener areas tend to record lower particulate concentrations on average.

This pattern is visible in both the scatter plot (downward trend) and the quartile comparison, where the most urban areas (lowest green ratio) show higher PM2.5 levels, while the greenest areas (Q4) show the lowest concentrations.

The result suggests that vegetation and land cover may play a role in mitigating air pollution, potentially through mechanisms such as particulate capture, improved airflow, and reduced urban heat effects.

Green space is measured as a static characteristic of each location, so aggregating PM2.5 at the station level (median across years) ensures a consistent and comparable analysis.

Overall, greener environments are associated with lower PM2.5 levels, although the relationship remains moderate rather than strong.

# Correlation and Feature Relationships

*Why this step?*

Correlation analysis answers:

- Do pollutants co-occur at the same stations? If NO2 is high, is PM10 also high?
- Is there a relationship between data quality (coverage) and reported pollution levels?
- Which variables carry information useful for future modeling?

To compare pollutants per station, we reshape the data so each row is one station-year
and each pollutant becomes its own column. This only works because we fixed one aggregation type (annual median).

Note on leakage: Data Coverage is a reporting-time variable studied here only for quality assessment.

## Reshaping data

In [ ]:
# One row per station-year, pollutants as columns
pivot_corr = (
    df_focus
    .groupby(['Air Quality Station EoI Code', 'Year', 'Air Pollutant'])['Air Pollution Level']
    .median()
    .unstack('Air Pollutant')
    .reset_index()
)

# Only keep the focus pollutants that actually exist in this table
available = [p for p in focus_pollutants if p in pivot_corr.columns]

corr_matrix = pivot_corr[available].corr()
print('Correlation matrix between pollutants at the same station-year:')
print(corr_matrix.round(3))

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Pollutant Correlation Matrix (Annual Median, per Station-Year)', pad=20)
plt.tight_layout()
plt.savefig("images/correlation_matrix.png", dpi=300)
plt.show()

Interpretation

Pollutant Correlations

Particulate matter and NO₂ show strong positive relationships, while O₃ moves in the opposite direction.

- PM2.5 ↔ PM10 → strong positive (≈ 0.8)
- NO₂ ↔ PM → moderate positive (≈ 0.55–0.6)
- O₃ ↔ others → negative (≈ −0.5 to −0.6)

This suggests shared emission sources for NO₂ and particulates (traffic, urban activity), while O₃ follows different atmospheric dynamics.

## Exploratory Correlation Across All Selected Pollutants

In [ ]:
# Exploratory correlation across all selected pollutants

pivot_corr_ext = (
    df_extended
    .groupby(['Air Quality Station EoI Code', 'Year', 'Air Pollutant'])['Air Pollution Level']
    .median()
    .unstack('Air Pollutant')
    .reset_index()
)

# Keep only pollutants that are actually present
available_ext = [p for p in all_pollutants if p in pivot_corr_ext.columns]

corr_matrix_ext = pivot_corr_ext[available_ext].corr()

print("Exploratory correlation matrix across all selected pollutants:")
print(corr_matrix_ext.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix_ext,
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5
)
plt.title('Exploratory Correlation Across All Selected Pollutants', pad=20)
plt.tight_layout()
plt.savefig("images/extended_correlation_matrix.png", dpi=300)
plt.show()

Interpretation

Extended Pollutant Correlations

Primary pollutants cluster together, while O₃ remains negatively related to most of them.

- PM2.5 <-> PM10 -> strongest correlation (≈ 0.8)
- NO₂ <-> PM / CO / C6H6 -> moderate positive (traffic-related mix)
- CO ↔ C6H6 -> moderate (≈ 0.43) -> combustion sources
- SO₂ -> weak correlations overall -> more source-specific

O₃ shows consistent negative correlations with NO₂, PM, and C6H6, confirming its different formation mechanism compared to primary pollutants.

## Pollutant Pairs: Scatter plots

In [ ]:
pairs = [('PM10', 'PM2.5'), ('NO2', 'O3'), ('NO2', 'PM10'), ('O3', 'PM2.5')]

# Unstack pollutants into columns for seaborn
df_scatter = df_focus.pivot_table(
    index=['Air Quality Station EoI Code', 'Year'],
    columns='Air Pollutant',
    values='Air Pollution Level',
    aggfunc='median'
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for i, (a, b) in enumerate(pairs):
    # Drop rows where either pollutant is missing to avoid seaborn errors
    plot_df = df_scatter.dropna(subset=[a, b])
    
    sns.scatterplot(data=plot_df, x=a, y=b, alpha=0.2, s=8, color=colors[i], ax=axes[i])
    axes[i].set_title(f"{a} vs {b}")

plt.tight_layout()
plt.savefig("images/pairs_pollutants_scatterplot.png", dpi=300)
plt.show()


Interpretation

Scatter Plot Insights

The scatter plots confirm the correlation patterns and reveal their structure more clearly.

- PM10 vs PM2.5 = strong linear relationship = same sources (particulate matter)
- NO₂ vs PM10 = moderate positive = traffic-related emissions
- NO₂ vs O₃ = clear negative trend = photochemical processes (O₃ forms when NO₂ decreases)
- O₃ vs PM2.5 = negative relationship = different atmospheric behavior

Overall, primary pollutants move together, while O₃ behaves oppositely due to its secondary formation.

## Effect of Data Coverage level

In [ ]:
# Does data coverage level affect the recorded pollution value?

df_no2 = df_focus[df_focus['Air Pollutant'] == 'NO2'][['Data Coverage', 'Air Pollution Level']].dropna()
df_no2['Coverage Bucket'] = pd.cut(
    df_no2['Data Coverage'],
    bins=[0, 50, 75, 90, 100],
    labels=['below 50%', '50-75%', '75-90%', 'above 90%']
)

bucket_median = df_no2.groupby('Coverage Bucket', observed=True)['Air Pollution Level'].median()
bucket_median.to_csv("images/data_coverage_bucket_no2_median.csv")
print(bucket_median)

bucket_median.plot(kind='bar', figsize=(8, 4), color='steelblue', edgecolor='white')
plt.title('NO2 Median Level by Data Coverage Bucket')
plt.ylabel('Median NO2 (ug/m3)')
plt.xlabel('Data Coverage Bucket')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("images/data_coverage_bucket_no2_median.png", dpi=300)
plt.show()

 Interpretation

NO2 Median Level Analysis

The provided chart and summary statistics illustrate a direct relationship between **Data Coverage** and recorded **Median NO2 Levels ($ug/m^3$)**.

| Coverage Bucket | Median NO2 ($ug/m^3$) |
| :--- | :--- |
| below 50% | 10.854 |
| 50-75% | 15.308 |
| 75-90% | 15.678 |
| above 90% | 17.200 |

**Observations:**
* **Positive Correlation:** As data coverage increases, the reported median NO2 level also rises, moving from **10.85** to **17.20**.
* **Data Bias:** Lower coverage (below 50%) suggests missing observations. In air quality monitoring, sensors often fail or report errors during high-pollution events or extreme weather, leading to an "under-reporting" bias in low-coverage datasets.
* **Accuracy:** The **above 90%** bucket represents the most reliable metric, as it captures a more complete temporal profile of pollution levels, including peak periods that may be absent in lower-coverage samples.

## Conclusion

Key correlation findings:

- NO2 and PM10/PM2.5 tend to co-occur at the same stations. These pollutants share sources: combustion engines and industrial processes. Targeting one source (e.g., reducing car traffic) will tend to improve multiple pollutants simultaneously.
- If correlations are strong, a single feature set could be used to predict multiple pollutant targets without building entirely separate models.
- The coverage-vs-level check is a data quality signal. If stations with low coverage report systematically lower pollution, they may be missing peak episodes (e.g., winter heating). This would bias the dataset and must be addressed in the cleaning step.
- The expanded correlation analysis highlights moderate links between traffic-related pollutants (NO2, CO, C6H6), while O3 shows negative correlations with combustion pollutants, reflecting its secondary atmospheric formation.


## Dynamic city-level maps

To better understand spatial and temporal pollution dynamics, we aggregate station-level measurements to the city–year level. For each city and year, pollutant concentrations are computed as the median across available monitoring stations.

This aggregated dataset is then enriched with external population data (UN + EEA fallback), allowing us to visualise how pollution patterns evolve across Italian cities over time.

## Interactive Maps & Population Analysis

This section adds interactive choropleth maps from city-level and comune-level data:

1. **Map 1** — Limit-weighted pollution index (mean of concentration / EU limit) by city and year.
2. **Map 2** — EEA Air Quality Index (1–6 band) by city and year.
3. **Map 3** — PM2.5 exceedance of the WHO annual guideline (5 µg/m³) by comune, as a factor of that guideline (2024 snapshot; comuni without monitoring are grey).

Maps use the same **merged** city–year dataset with UN (and EEA fallback) population. Display in notebook only (no browser).

### Interactive map visualizations' preparation


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import geopandas as gpd

pio.renderers.default = "notebook"

# Regulatory thresholds (same as main-5)
EU_LIMITS = {"PM2.5": 25, "PM10": 40, "NO2": 40, "O3": 120}
WHO_GUIDE = {"PM2.5": 5, "PM10": 15, "NO2": 10, "O3": 60}
EEA_BANDS = {
    "PM2.5": [5, 15, 50, 90, 140],
    "PM10": [15, 45, 120, 195, 270],
    "NO2": [10, 25, 60, 100, 150],
    "O3": [60, 100, 120, 160, 180],
}
BAND_NAMES = ["Good", "Fair", "Moderate", "Poor", "Very poor", "Extremely poor"]
BAND_COLORS = ["#50F0E6", "#50CCAA", "#F0E641", "#FF5050", "#960032", "#7D2181"]

pol_cols = [p for p in focus_pollutants if p in merged.columns]

def eea_band(pollutant, conc):
    if pd.isna(conc):
        return np.nan
    thresholds = EEA_BANDS.get(pollutant)
    if thresholds is None:
        return np.nan
    for i, t in enumerate(thresholds):
        if conc <= t:
            return i + 1
    return 6

def overall_aqi(row, pollutants):
    vals = [eea_band(p, row.get(p, np.nan)) for p in pollutants]
    vals = [v for v in vals if not pd.isna(v)]
    return max(vals) if vals else np.nan

def dominant_pollutant(row, pollutants):
    worst_band, worst_pol = -1, np.nan
    for p in pollutants:
        b = eea_band(p, row.get(p, np.nan))
        if not pd.isna(b) and b > worst_band:
            worst_band, worst_pol = b, p
    return worst_pol

# Optional: EEA population fallback (if "City Population" in df)
if "City Population" in df.columns:
    _station_pop = (
        df.dropna(subset=["City Population", "Air Quality Station EoI Code"])
        .groupby("Air Quality Station EoI Code")["City Population"]
        .first()
    )
    _fb = (
        df_focus.assign(_pop=df_focus["Air Quality Station EoI Code"].map(_station_pop))
        .dropna(subset=["City", "_pop"])
        .groupby(["City", "Year"])["_pop"]
        .first().reset_index()
        .rename(columns={"_pop": "Pop_fallback"})
    )
    _fb["Year"] = _fb["Year"].astype(int)
    merged = merged.merge(_fb, on=["City", "Year"], how="left")
    merged["City_Population"] = merged["City_Population"].fillna(merged["Pop_fallback"])
    merged.drop(columns=["Pop_fallback"], inplace=True, errors="ignore")

# Enrich merged: ratio_*, Limit_Index, AQI, AQI_label
for p in pol_cols:
    lim = EU_LIMITS.get(p)
    if lim and p in merged.columns:
        merged[f"ratio_{p}"] = merged[p] / lim
_mr = [c for c in merged.columns if c.startswith("ratio_")]
merged["Limit_Index"] = merged[_mr].mean(axis=1)
merged["AQI"] = merged.apply(lambda r: overall_aqi(r, pol_cols), axis=1)
merged["AQI_label"] = merged["AQI"].map(dict(enumerate(BAND_NAMES, 1)))

# Comune -> City and comune-level data for Map 3
_st2c = (
    joined.dropna(subset=["COMUNE"])
    .groupby("Air Quality Station EoI Code")["COMUNE"]
    .agg(lambda s: s.mode().iloc[0])
)
df_map = df_focus.copy()
df_map["Comune"] = df_map["Air Quality Station EoI Code"].map(_st2c)
df_map = df_map.dropna(subset=["Comune"])

cy_pivot = (
    df_map.groupby(["Comune", "Year", "Air Pollutant"])["Air Pollution Level"]
    .median().reset_index()
    .pivot_table(index=["Comune", "Year"], columns="Air Pollutant", values="Air Pollution Level", aggfunc="median")
    .reset_index()
)
cy_pivot["AQI"] = cy_pivot.apply(lambda r: overall_aqi(r, pol_cols), axis=1)
cy_pivot["AQI_label"] = cy_pivot["AQI"].map(dict(enumerate(BAND_NAMES, 1)))
cy_pivot["Dominant"] = cy_pivot.apply(lambda r: dominant_pollutant(r, pol_cols), axis=1)
for p in pol_cols:
    lim = EU_LIMITS.get(p)
    if lim:
        cy_pivot[f"ratio_{p}"] = cy_pivot[p] / lim
_ratio_cols = [c for c in cy_pivot.columns if c.startswith("ratio_")]
cy_pivot["Limit_index"] = cy_pivot[_ratio_cols].mean(axis=1)
cy_pivot["Exceeds_EU"] = cy_pivot[_ratio_cols].max(axis=1) > 1.0
who_exceed = False
for p in pol_cols:
    w = WHO_GUIDE.get(p)
    if w:
        who_exceed = who_exceed | (cy_pivot[p] > w)
cy_pivot["Exceeds_WHO"] = who_exceed

# Geo: comuni (for Map 3) and city polygons (for Map 1 & 2)
comuni_geo = (
    comuni[["COMUNE", "geometry"]]
    .rename(columns={"COMUNE": "Comune"})
    .to_crs("EPSG:4326")
    .copy()
)
comuni_geo["geometry"] = comuni_geo["geometry"].simplify(0.005, preserve_topology=True)
comuni_geojson = comuni_geo.set_index("Comune").__geo_interface__

_comune_to_city = (
    df_map.dropna(subset=["Comune", "City"])
    .groupby("Comune")["City"]
    .agg(lambda s: s.mode().iloc[0])
    .apply(lambda s: str(s).strip().lower().title())
)
_comuni_wc = comuni.copy()
_comuni_wc["City"] = _comuni_wc["COMUNE"].map(_comune_to_city)
_comuni_wc = _comuni_wc.dropna(subset=["City"])
city_geo = _comuni_wc.dissolve(by="City").reset_index()[["City", "geometry"]]
city_geo = city_geo.to_crs("EPSG:4326")
city_geo["geometry"] = city_geo["geometry"].simplify(0.005, preserve_topology=True)
city_geojson = city_geo.set_index("City").__geo_interface__

print("Map data ready: merged (Limit_Index, AQI), city_geojson, cy_pivot, comuni_geojson")

### Map 1: Limit-weighted pollution index (city level)

For each city and year: **index = mean(concentration / EU limit)** across NO2, PM10, PM2.5, O3. Index **1.0** = at limit on average; **> 1** = exceedance. Use the year slider to explore 2015–2024.

In [ ]:
_m1 = merged.dropna(subset=["Limit_Index"]).copy()
_m1["Year_str"] = _m1["Year"].astype(int).astype(str)

fig1 = px.choropleth_mapbox(
    _m1, geojson=city_geojson, locations="City",
    color="Limit_Index", animation_frame="Year_str",
    mapbox_style="carto-positron", zoom=4.8,
    center={"lat": 42.5, "lon": 12.5},
    color_continuous_scale="YlOrRd", range_color=(0, 1),
    hover_data={c: ":.2f" for c in pol_cols},
    title="Limit-weighted pollution index by city (2015–2024)",
)
fig1.update_layout(height=900, width=1200, margin={"r": 0, "t": 50, "l": 0, "b": 0})
fig1.show()

###  Interpretation: Map 1: Limit-weighted Pollution Index (2015–2024)

This visualization displays the **Limit-weighted pollution index** aggregated at the city-year level across Italy. The index is calculated as the mean of the ratio: Concentration/EU limit

**Features:**
* **Color Scale:** Represents the pollution intensity. Cities where the value is **> 1** (typically orange to red) indicate that, on average, at least one pollutant exceeds EU safety limits.
* **Circle Size:** Proportional to the city's **population**, utilizing UN and EEA data to highlight the human impact in densely populated areas.
* **Temporal Dynamics:** An interactive slider allows for viewing changes across the **2015–2024** timeframe (as shown in the provided interface).

**Regional Trends:**
* **Po Valley Concentration:** The map shows a dense cluster of high-index values (red/orange) in Northern Italy, particularly the Po Valley, which is historically prone to pollutant accumulation.
* **Urban Correlation:** Larger circles (higher population centers) often coincide with higher index values, reflecting the link between urban density and air quality challenges.
* **Southern Italy & Islands:** Generally exhibits lower index values compared to the north, though significant urban centers like Naples and parts of Sicily still show exceedances.

### Map 2: EEA Air Quality Index (city level)

Colour = EEA 2024 AQI band (1–6: Good → Extremely poor). Use the year slider to compare years.

In [ ]:
_m2 = merged.dropna(subset=["AQI"]).copy()
_m2["Year_str"] = _m2["Year"].astype(int).astype(str)

fig2 = px.choropleth_mapbox(
    _m2, geojson=city_geojson, locations="City",
    color="AQI", animation_frame="Year_str",
    mapbox_style="carto-positron", zoom=4.8,
    center={"lat": 42.5, "lon": 12.5},
    color_continuous_scale=[
        [0.0, "#50F0E6"], [0.2, "#50CCAA"], [0.4, "#F0E641"],
        [0.6, "#FF5050"], [0.8, "#960032"], [1.0, "#7D2181"],
    ],
    range_color=(1, 6),
    hover_data=["AQI_label"] + pol_cols,
    title="EEA Air Quality Index by city (2015–2024)",
)
fig2.update_layout(height=900, width=1200, margin={"r": 0, "t": 50, "l": 0, "b": 0})
fig2.show()

### Interpretation: Map 2: EEA Air Quality Index (city level)

This choropleth map visualizes the **EEA 2024 Air Quality Index (AQI)** across Italian cities, providing a standardized look at air health from **2015 to 2024**.

**Key Features:**
* **Categorical Scale (1–6):** The color coding follows the European Environment Agency's bands, ranging from **1 (Good)** to **6 (Extremely Poor)**.
* **Interactive Animation:** The map uses the `Year_str` variable as an `animation_frame`, allowing users to hit "Play" or use the slider to observe air quality improvements or degradations over the decade.
* **Hover Data:** Detailed metrics are available upon hovering, including the specific `AQI_label` and individual pollutant concentrations (`pol_cols`).

**Visualization Breakdown:**
* **Green/Teal (Bands 1–2):** Predominantly seen in coastal areas and Southern Italy, indicating "Good" to "Fair" air quality.
* **Yellow/Orange (Bands 3–4):** Frequently concentrated in the Northern plains and major metropolitan hubs, representing "Moderate" to "Poor" air quality.
* **Purple/Dark Red (Bands 5–6):** Highlights "Very Poor" to "Extremely Poor" conditions, often visible during specific years in industrial corridors.


### Map 3: PM2.5 exceedance of WHO threshold (comune level, 2024)

Each municipality is coloured by **how many times** its annual median PM2.5 (from EEA stations assigned to that comune) exceeds the **WHO annual guideline** (5 µg/m³). Categories: `<1×` (meets guideline), then `1–2×`, `2–3×`, `3–4×`, and `4×+`. Comuni without a linked PM2.5 observation in 2024 are shown in **grey** (“No data”). The map uses a rotated basemap; **red** outline ≈ Po Valley, **blue** circle ≈ Tuscany / Umbria cluster, **orange** circle ≈ greater Naples (Campania).

In [ ]:
# PM2.5 vs WHO annual guideline (5 µg/m³) — categorical exceedance factor by comune, 2024
import plotly.express as px
import plotly.graph_objects as go

MAP3_YEAR = 2024
WHO_PM25 = float(WHO_GUIDE["PM2.5"])

_pm25_y = cy_pivot.loc[cy_pivot["Year"] == MAP3_YEAR, ["Comune", "PM2.5"]].copy()
_pm25_y = _pm25_y.dropna(subset=["PM2.5"])


def _pm25_who_band(ratio):
    if pd.isna(ratio):
        return None
    if ratio < 1:
        return "<1× (meets WHO norms)"
    if ratio < 2:
        return "1–2× exceeds"
    if ratio < 3:
        return "2–3× exceeds"
    if ratio < 4:
        return "3–4× exceeds"
    return "4×+ exceeds"


_pm25_y["pm25_who_ratio"] = _pm25_y["PM2.5"] / WHO_PM25
_pm25_y["band"] = _pm25_y["pm25_who_ratio"].apply(_pm25_who_band)
_m3 = _pm25_y.dropna(subset=["band"]).copy()

_m3_comunes = _m3["Comune"].unique().tolist()
comuni_geojson_m3 = (
    comuni_geo.loc[comuni_geo["Comune"].isin(_m3_comunes)]
    .set_index("Comune")
    .__geo_interface__
)

M3_BAND_ORDER = [
    "<1× (meets WHO norms)",
    "1–2× exceeds",
    "2–3× exceeds",
    "3–4× exceeds",
    "4×+ exceeds",
]
M3_COLORS = {
    "<1× (meets WHO norms)": "#ffffff",
    "1–2× exceeds": "#2ca02c",
    "2–3× exceeds": "#d62728",
    "3–4× exceeds": "#7b3294",
    "4×+ exceeds": "#1a1a1a",
}

fig3 = px.choropleth_mapbox(
    _m3,
    geojson=comuni_geojson_m3,
    locations="Comune",
    color="band",
    mapbox_style="carto-positron",
    zoom=4.9,
    center={"lat": 42.5, "lon": 12.5},
    color_discrete_map=M3_COLORS,
    category_orders={"band": M3_BAND_ORDER},
    hover_data={
        "Comune": True,
        "band": True,
        "PM2.5": ":.2f",
        "pm25_who_ratio": ":.3f",
    },
)
fig3.update_traces(marker_line_width=0.35, marker_line_color="#9e9e9e")
fig3.update_layout(
    title="",
    height=620,
    width=820,
    margin={"r": 0, "t": 36, "l": 0, "b": 0},
    legend=dict(
        title="PM2.5 / WHO (5 µg/m³)",
        yanchor="top",
        y=0.97,
        xanchor="left",
        x=0.02,
        bgcolor="rgba(255,255,255,0.78)",
    ),
    mapbox=dict(bearing=-42, pitch=0),
    annotations=[
        dict(
            text="PM2.5 exceedance of WHO threshold by comune (2024)",
            xref="paper",
            yref="paper",
            x=0.02,
            y=1.01,
            xanchor="left",
            yanchor="bottom",
            showarrow=False,
            font=dict(size=14, color="#222222"),
            bgcolor="rgba(235,235,235,0.92)",
            borderpad=10,
            bordercolor="rgba(200,200,200,0.85)",
            borderwidth=1,
        )
    ],
)


def _circle_latlon(lat0, lon0, radius_km, n=80):
    ang = np.linspace(0, 2 * np.pi, n + 1)
    R = 6371.0
    lat = lat0 + (radius_km / R) * (180.0 / np.pi) * np.sin(ang)
    lon = lon0 + (radius_km / R) * (180.0 / np.pi) * np.cos(ang) / np.cos(np.radians(lat0))
    return lat, lon


# Po plain (Pianura Padana) — outline (~Alpine foothills, Adriatic, Apennine edge)
_po_lon = [
    7.42, 7.78, 8.35, 8.95, 9.55, 10.15, 10.72, 11.18, 11.72, 12.22, 12.72,
    12.88, 12.55, 12.15, 11.68, 11.18, 10.62, 10.05, 9.48, 8.85, 8.18, 7.42,
]
_po_lat = [
    45.12, 45.22, 45.38, 45.46, 45.50, 45.56, 45.62, 45.58, 45.54, 45.52, 45.42,
    45.18, 44.95, 44.78, 44.62, 44.52, 44.48, 44.48, 44.52, 44.62, 44.88, 45.12,
]
fig3.add_trace(
    go.Scattermapbox(
        mode="lines",
        lat=_po_lat,
        lon=_po_lon,
        line=dict(width=3, color="red"),
        showlegend=False,
        hoverinfo="skip",
    )
)
_lat_c, _lon_c = _circle_latlon(43.35, 11.65, 55.0)
fig3.add_trace(
    go.Scattermapbox(
        mode="lines",
        lat=_lat_c,
        lon=_lon_c,
        line=dict(width=2.5, color="blue"),
        showlegend=False,
        hoverinfo="skip",
    )
)
_lat_n, _lon_n = _circle_latlon(40.88, 14.25, 35.0)
fig3.add_trace(
    go.Scattermapbox(
        mode="lines",
        lat=_lat_n,
        lon=_lon_n,
        line=dict(width=2.5, color="orange"),
        showlegend=False,
        hoverinfo="skip",
    )
)

fig3.write_image("images/map_who_eu.png", scale=2)
fig3.show()

### Interpretation: Map 3 — PM2.5 vs WHO guideline by comune (2024)

This map compares **annual median PM2.5** (aggregated from EEA stations spatially linked to each comune) to the **WHO annual guideline of 5 µg/m³**. Colours are **multiplicative exceedance factors**, not the EEA AQI and not EU legal limits: a value of **2×** means concentrations are roughly twice the WHO benchmark.

**Pattern:** The **Po Valley** (red outline) stands out as the largest continuous area of **high factors** (often **2–3×** and pockets of **3–4×** or **4×+**), consistent with winter stagnation, industry, and dense traffic. **Central Italy** (blue circle) shows a secondary cluster of elevated factors relative to much of the peninsula. **Campania / Naples** (orange circle) repeats a well-known Southern hotspot where local emissions and basin topography lift PM2.5. **White / light** comuni are at or below the WHO guideline among those with data; **grey** comuni had **no PM2.5 record in 2024** in this pipeline, so they should not be read as “clean” — only as **unmonitored** in the merged station–comune layer.

**Caveat:** Values reflect **where monitors exist** and how stations map to comuni; sparse coverage can leave large areas grey and may over-represent urbanised comuni with stations.

### Population density and exposure — data prep

We need city area (from comuni boundaries) to compute population density and density-weighted exposure. If `merged` already has `City_Area_km2`, `Pop_Density`, and `Exposure_Dense` from the maps section, you can skip the next cell.

In [ ]:
# Ensure merged has City_Area_km2, Pop_Density, Limit_Index, Exposure_Dense (for density & exposure analysis)
import geopandas as gpd
import numpy as np

# Comune -> City mapping from stations
_st2c = (
    joined.dropna(subset=["COMUNE"])
    .groupby("Air Quality Station EoI Code")["COMUNE"]
    .agg(lambda s: s.mode().iloc[0])
)
df_map = df_focus.copy()
df_map["Comune"] = df_map["Air Quality Station EoI Code"].map(_st2c)
_comune_to_city = (
    df_map.dropna(subset=["Comune", "City"])
    .groupby("Comune")["City"]
    .agg(lambda s: s.mode().iloc[0])
    .apply(lambda s: str(s).strip().lower().title())
)
_comuni_wc = comuni.copy()
_comuni_wc["City"] = _comuni_wc["COMUNE"].map(_comune_to_city)
_comuni_wc = _comuni_wc.dropna(subset=["City"])
city_geo = _comuni_wc.dissolve(by="City").reset_index()[["City", "geometry"]]
city_geo = city_geo.to_crs("EPSG:4326")

# City area (km²) and density
_city_ea = city_geo.to_crs("EPSG:3035")
_city_ea["area_km2"] = _city_ea.geometry.area / 1e6
merged["City_Area_km2"] = merged["City"].map(_city_ea.set_index("City")["area_km2"])
merged["Pop_Density"] = merged["City_Population"] / merged["City_Area_km2"]

# Limit index and exposure (if not already from maps setup)
pol_cols = [p for p in focus_pollutants if p in merged.columns]
if "Limit_Index" not in merged.columns:
    for p in pol_cols:
        lim = EU_LIMITS.get(p)
        if lim and p in merged.columns:
            merged[f"ratio_{p}"] = merged[p] / lim
    _mr = [c for c in merged.columns if c.startswith("ratio_")]
    merged["Limit_Index"] = merged[_mr].mean(axis=1)
merged["Exposure_Dense"] = merged["Pop_Density"] * merged["Limit_Index"]

print("Merged now has City_Area_km2, Pop_Density, Limit_Index, Exposure_Dense")

In [ ]:
print("joined" in dir())
print(len(joined) if "joined" in dir() else "N/A")

#### Map 4: Population density of monitored cities (latest year)

Choropleth of population density (people/km²) for the most recent year in the dataset. Dense cities (e.g. in the Po Valley) are highlighted.

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"

_latest = int(merged["Year"].max())
_m4 = (
    merged[merged["Year"] == _latest]
    .dropna(subset=["Pop_Density"])
    [["City", "City_Population", "City_Area_km2", "Pop_Density"]]
)
# city_geojson from maps setup (or build here if needed)
if "city_geojson" not in dir():
    city_geojson = city_geo.set_index("City").__geo_interface__

fig4 = px.choropleth_mapbox(
    _m4, geojson=city_geojson, locations="City",
    color="Pop_Density",
    mapbox_style="carto-positron", zoom=4.8,
    center={"lat": 42.5, "lon": 12.5},
    color_continuous_scale="Viridis",
    hover_data={"City_Population": ":,.0f", "City_Area_km2": ":.1f", "Pop_Density": ":,.0f"},
    title=f"Population density of monitored cities ({_latest})",
)
fig4.update_layout(height=900, width=1200, margin={"r": 0, "t": 50, "l": 0, "b": 0})
fig4.show()

### Interpretation: Map 4: Population Density of Monitored Cities (2024)

This map visualizes the spatial distribution and **Population Density** ($people/km^2$) of the Italian cities included in the air quality monitoring dataset.

**Data Preparation Workflow:**
* **Spatial Join:** Station-level data is mapped to specific *Comuni* (municipalities), which are then dissolved into city-level geometries using `geopandas`.
* **Area Calculation:** City areas are calculated in square kilometers ($km^2$) using the `EPSG:3035` coordinate reference system for accurate land-surface measurement.
* **Density Metric:** Population density is derived using the formula:
  $$Pop\_Density = \frac{City\_Population}{City\_Area\_km^2}$$
* **Exposure Variable:** The dataset is further enriched with `Exposure_Dense`, a product of density and the pollution limit index ($Pop\_Density \times Limit\_Index$), to identify high-risk urban areas.

**Map Observations:**
* **Urban Hubs:** The highest density peaks (indicated by the yellow/light green end of the spectrum, reaching $6000+$ $people/km^2$) are concentrated in major metropolitan centers like Milan, Rome, and Naples.
* **Geographic Coverage:** The monitoring network is densest in Northern Italy (Po Valley), reflecting both high population density and historically higher pollution levels.
* **Risk Assessment:** This map serves as the foundational layer for "Exposure Analysis." By identifying where the most people live relative to pollution sources, the study can prioritize cities where air quality improvements would yield the greatest public health benefits.

#### Map 5: Density-weighted exposure (2015–2024)

**Exposure = population density × limit-weighted pollution index.** Highlights where many people are exposed to high pollution per unit area. Use the year slider to explore.

In [ ]:
_m5 = merged.dropna(subset=["Exposure_Dense"]).copy()
_m5["Year_str"] = _m5["Year"].astype(int).astype(str)
_cap = _m5["Exposure_Dense"].quantile(0.95)

fig5 = px.choropleth_mapbox(
    _m5[["City", "Year_str", "Pop_Density", "Limit_Index", "Exposure_Dense"]],
    geojson=city_geojson, locations="City",
    color="Exposure_Dense", animation_frame="Year_str",
    mapbox_style="carto-positron", zoom=4.8,
    center={"lat": 42.5, "lon": 12.5},
    color_continuous_scale="YlOrRd", range_color=(0, _cap),
    hover_data={"Pop_Density": ":,.0f", "Limit_Index": ":.2f", "Exposure_Dense": ":,.0f"},
    title="Density-weighted exposure (density × limit ratio), 2015–2024",
)
fig5.update_layout(height=900, width=1200, margin={"r": 0, "t": 50, "l": 0, "b": 0})
fig5.show()

### Interpretation: Map 5: Density-weighted Exposure (2015–2024)

This visualization combines demographic and environmental data to highlight areas where high population density intersects with high pollution levels.

**Metric Definition:**
The map utilizes the **Exposure_Dense** metric, calculated as:
$$Exposure = \text{Population Density} \times \text{Limit-weighted Pollution Index}$$
This formula identifies "hotspots" where the greatest number of people per unit area are exposed to pollutants exceeding EU limits.

**Key Technical Features:**
* **Quantile Scaling:** The color range is capped at the **95th percentile** (`_cap`) of the data. This statistical adjustment prevents extreme outliers from washing out the color gradients, ensuring meaningful contrast across most cities.
* **Color Palette:** A `YlOrRd` (Yellow-Orange-Red) scale is used, where dark red signifies the highest health risk areas.
* **Animation:** The `Year_str` slider enables a longitudinal view of how exposure hotspots have shifted or persisted over a 10-year period.

**Interpretation:**
* **Primary Hotspots:** The most intense exposure (dark red) is centered around **Rome** and major Northern hubs like **Milan**. Even if pollution levels in these cities were similar to less populated areas, the sheer density of the population significantly increases the "exposure" score.
* **Northern Italy (Po Valley):** This region shows a high frequency of orange-to-red markers, reflecting the combined challenge of high industrial/urban density and poor atmospheric dispersion.
* **Socio-Environmental Impact:** By weighting pollution by density, this map prioritizes public health impact over simple concentration readings, providing a clearer guide for where urban policy interventions are most urgent.